# UWATR — build the frozen feature dataset

One run produces the shared feature space for all three corpora, so no later
experiment ever re-reads 30 GB of wav or re-derives an STFT.

**Inputs** (already attached): `deepship-raw`, `oceanship`, `workspace`.

**Output**: `/kaggle/working/uwatr-features`, about **14 GB** — log-mel plus
linear spectrogram for Oceanship (53,770 rows), DeepShip (~33,710) and ShipsEar
(2,223), each with an `index.csv` carrying the keys the frozen splits join on.

**Accelerator**: a GPU is *not* needed. The build runs on **CPU on purpose** —
a frozen artifact should not depend on which accelerator was allocated that day.
Expect roughly 20-40 minutes. Use `Save Version -> Save & Run All (Commit)` so
it runs in the background.

**Then**: open the finished version, go to the **Output** tab and click
**Create Dataset** (this is the official route — no download/re-upload needed).
Pick the whole `uwatr-features` folder. The new dataset becomes the only input
every modelling notebook needs.

> If a corpus is missing from the inputs, this notebook says so and builds the
> rest instead of aborting. Read the `inputs` and `DEGRADED` lines in cell 4.
> Kaggle expands archives on ingest, so every input is accepted both as the
> archive and as the tree it was expanded into — see the last cell.


In [ ]:
# --------------------------------------------------------------------------- #
# 1. environment + the list of mounted datasets
#
# Kaggle mounts a dataset under its slug, either as /kaggle/input/<slug> or as
# /kaggle/input/datasets/<owner>/<slug>. Enumerating those two levels is exact
# and cheap. Hunting for marker files inside them is not: an rglob into the
# Oceanship dataset walks ~54,000 entries, and doing that once per candidate cost
# 15 minutes on the first run.
#
# If a dataset is attached under another name, set MANUAL below to a path this
# cell prints and re-run.
# --------------------------------------------------------------------------- #
import shutil
import subprocess
import sys
from pathlib import Path

for mod in ("soundfile", "scipy"):
    try:
        __import__(mod)
    except ImportError:
        print(f"installing {mod} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)

import numpy as np
import soundfile as sf

try:
    import torch
    print("torch", torch.__version__, "cuda:", torch.cuda.is_available())
except ImportError:
    torch = None
    print("torch not available - fine, the build is CPU by design")

OUT = Path("/kaggle/working/uwatr-features")
OUT.mkdir(parents=True, exist_ok=True)
SCRIPTS = Path("/kaggle/working/scripts")
SCRIPTS.mkdir(parents=True, exist_ok=True)

INPUT = Path("/kaggle/input")

# escape hatch: MANUAL["workspace"] = "/kaggle/input/datasets/adnygong/uwatr-workspace"
MANUAL = {"deepship-raw": None, "oceanship": None, "workspace": None}


def mounted_datasets():
    """Every mounted dataset folder. Both layouts are at most one level deep."""
    found = []
    for base, flat in ((INPUT, True), (INPUT / "datasets", False)):
        if not base.is_dir():
            continue
        for child in sorted(base.iterdir()):
            if not child.is_dir() or (flat and child.name == "datasets"):
                continue
            if flat:
                found.append(child)
            else:
                found += [d for d in sorted(child.iterdir()) if d.is_dir()]
    seen, uniq = set(), []
    for d in found:
        if str(d) not in seen:
            seen.add(str(d))
            uniq.append(d)
    return uniq


def top_entries(d, n=14):
    try:
        return [p.name for p in sorted(d.iterdir())][:n]
    except Exception as exc:  # unreadable mount
        return [f"<unreadable: {exc}>"]


MOUNTED = mounted_datasets()
print()
print("=" * 88)
print("mounted datasets")
print("=" * 88)
for d in MOUNTED:
    print(f"  {d}")
    print(f"      {top_entries(d)}")
if not MOUNTED:
    print("  (nothing under /kaggle/input - attach the datasets via Add Input)")


In [ ]:
%%writefile /kaggle/working/scripts/build_features.py
#!/usr/bin/env python3
"""W2-03  Build the frozen feature dataset for every UATR corpus in one pass.

Three corpora, one representation space
---------------------------------------
Oceanship, DeepShip and ShipsEar differ in container, sample rate and labelling
scheme, but they have to land in the *same* feature space, otherwise the
cross-corpus experiments compare pipelines instead of data.  So there is exactly
one set of STFT / filterbank / dB constants in this file and every job uses it.

Output per corpus (row-aligned, so a row index is a stable clip id):

    logmel.npy   (N, 128, 313) fp16   log-mel, dB re clip peak, floored at -80
    spec.npy     (N, 128, 313) fp16   linear frequency, 128 pooled bands
    index.csv    row -> identity + grouping keys + measured source duration

Two representations on purpose: if the strict-protocol collapse shows up in mel
only, that is a feature-engineering artefact; if it shows up in both, it is a
property of the data.  The paper needs the second statement, so the ablation has
to be built in from the start.

Why a measurement pass before writing
-------------------------------------
The cache is frozen and mounted read-only by every later run, so its shape has
to be right the first time.  A measurement pass reads only wav headers and gives
the exact row count (clips longer than 5 s become several rows) plus the true
duration distribution -- which matters, because duration is itself class
correlated in Oceanship (Passenger median 3.67 s against Tug 5.00 s).  Without
an exact count we would have to allocate a guess and then rewrite an 8 GB
memmap just to trim it.

Why the duration is written into index.csv
------------------------------------------
Short clips are padded rather than dropped: dropping everything under half a
window would silently remove an eighth of Oceanship and, if the short clips
cluster in particular classes, that is a class-balance bug rather than a tidy-up.
The cost of padding is a confound (a 2 s clip padded to 5 s is 60% silence), so
the measured duration goes into the index and the ablation is one join away.

Resampling note: everything is resampled through one polyphase routine
(scipy.signal.resample_poly).  The earlier per-corpus scripts used decimate() for
DeepShip and resample_poly() for Oceanship, which are not bit-identical; a single
routine for all three corpora is what makes rows from different corpora
comparable.

Usage
-----
    # what would it do? (headers only, seconds)
    python w2_build_feature_dataset.py --job oceanship --probe \\
        --oceanship-audio /kaggle/input/oceanship \\
        --oceanship-inventory .../oceanship_inventory.csv.gz --out /kaggle/working/uwatr-features

    # build everything
    python w2_build_feature_dataset.py --job all --out /kaggle/working/uwatr-features ...

    # half-size run (mel only) if the working directory is tight
    ... --reps mel
"""

from __future__ import annotations

import argparse
import collections
import csv
import gzip
import io
import json
import os
import re
import shutil
import sys
import tarfile
import time
import zipfile
from pathlib import Path

import numpy as np

try:
    import soundfile as sf
except ImportError:  # pragma: no cover
    print("ERROR: needs soundfile", file=sys.stderr)
    raise

try:
    from scipy.signal import resample_poly
except ImportError:  # pragma: no cover
    resample_poly = None

try:
    import torch
except ImportError:  # pragma: no cover
    torch = None


# --------------------------------------------------------------------------- #
# frozen representation configuration
#
# These constants define the public artifact.  Do not tune them per corpus, and
# do not tune them to chase accuracy: a cache that changes between runs makes
# every ablated number unreproducible, which is the exact failure mode this
# project exists to complain about.
# --------------------------------------------------------------------------- #
TGT_SR = 16_000
SEG_S = 5.0
SEG_LEN = int(TGT_SR * SEG_S)          # 80_000 samples
N_FFT = 1024
HOP = 256
N_BINS = N_FFT // 2 + 1                # 513
N_MELS = 128
N_FRAMES = 1 + SEG_LEN // HOP          # 313
MEL_FMIN, MEL_FMAX = 20.0, 8000.0
TOP_DB = 80.0
MIN_KEEP_S = 1.0                       # below this there is too little signal to pad
BATCH = 64

BYTES_PER_ROW = N_MELS * N_FRAMES * 2  # fp16


# --------------------------------------------------------------------------- #
# shared DSP
# --------------------------------------------------------------------------- #
def _hz_to_mel(f: np.ndarray) -> np.ndarray:
    return 2595.0 * np.log10(1.0 + f / 700.0)


def _mel_to_hz(m: np.ndarray) -> np.ndarray:
    return 700.0 * (10.0 ** (m / 2595.0) - 1.0)


def build_filterbanks() -> tuple[np.ndarray, np.ndarray]:
    """Return (mel_fb, linear_fb), each (128, 513).

    Written by hand rather than taken from librosa so the frozen cache does not
    depend on whichever librosa the current Kaggle image happens to ship.
    """
    freqs = np.linspace(0.0, TGT_SR / 2, N_BINS)

    mel_pts = np.linspace(_hz_to_mel(np.array([MEL_FMIN]))[0],
                          _hz_to_mel(np.array([MEL_FMAX]))[0], N_MELS + 2)
    hz_pts = _mel_to_hz(mel_pts)
    mel_fb = np.zeros((N_MELS, N_BINS), dtype=np.float32)
    for i in range(N_MELS):
        lo, mid, hi = hz_pts[i], hz_pts[i + 1], hz_pts[i + 2]
        for j, f in enumerate(freqs):
            if lo <= f <= mid and mid > lo:
                mel_fb[i, j] = (f - lo) / (mid - lo)
            elif mid < f <= hi and hi > mid:
                mel_fb[i, j] = (hi - f) / (hi - mid)
    mel_fb *= (2.0 / (hz_pts[2:N_MELS + 2] - hz_pts[:N_MELS]))[:, None]

    # linear: average-pool 513 bins into 128 groups so both representations share
    # a shape and the ablation compares like with like
    lin_fb = np.zeros((N_MELS, N_BINS), dtype=np.float32)
    edges = np.linspace(0, N_BINS, N_MELS + 1).astype(int)
    for i in range(N_MELS):
        a, b = edges[i], max(edges[i + 1], edges[i] + 1)
        lin_fb[i, a:b] = 1.0 / (b - a)
    return mel_fb, lin_fb


def stft_power_batch(clips: np.ndarray, device: str) -> np.ndarray:
    """(B, SEG_LEN) -> (B, 513, N_FRAMES) power.

    Both branches must agree frame for frame: the cache is a public artifact, so
    a CPU build has to be comparable with a GPU build.  torch.stft(center=True)
    pads by n_fft//2 with 'reflect' and yields 1 + SEG_LEN//HOP = 313 frames;
    scipy with boundary=None/padded=False would yield 309, so the scipy branch
    pads identically first.  Getting this wrong once produced a 309-frame CPU
    cache and a 313-frame GPU cache from the same code.
    """
    win = np.hanning(N_FFT).astype(np.float32)
    if torch is not None and device != "cpu":
        t = torch.from_numpy(clips).to(device)
        w = torch.from_numpy(win).to(device)
        spec = torch.stft(t, n_fft=N_FFT, hop_length=HOP, win_length=N_FFT,
                          window=w, center=True, return_complex=True, pad_mode="reflect")
        return spec.abs().pow(2)[:, :, :N_FRAMES].cpu().numpy()

    from scipy.signal import stft as sp_stft

    pad = N_FFT // 2
    out = np.empty((clips.shape[0], N_BINS, N_FRAMES), dtype=np.float32)
    for i, c in enumerate(clips):
        c = np.pad(c, (pad, pad), mode="reflect")
        _, _, z = sp_stft(c, fs=TGT_SR, nperseg=N_FFT, noverlap=N_FFT - HOP,
                          window=win, boundary=None, padded=False)
        out[i] = (np.abs(z) ** 2)[:, :N_FRAMES]
    return out


def to_db(power: np.ndarray) -> np.ndarray:
    """Power -> dB relative to each clip's own peak, floored at -TOP_DB.

    Relative rather than absolute removes recorder gain, an instrument artifact,
    while keeping the within-clip dynamic range that carries the ship signature.
    Absolute level, which the attribution study needs as a range proxy, stays in
    the metadata.  Output range is exactly [-TOP_DB, 0].
    """
    peak = np.maximum(power.max(axis=(1, 2), keepdims=True), 1e-12)
    db = 10.0 * np.log10(np.maximum(power, peak * 1e-12) / peak)
    return np.maximum(db, -TOP_DB).astype(np.float32)


def to_mono_16k(x: np.ndarray, sr: int) -> np.ndarray:
    """Downmix to mono and resample to TGT_SR with one polyphase routine."""
    if x.ndim > 1 and x.shape[1] > 1:
        x = x.mean(axis=1)
    elif x.ndim > 1:
        x = x[:, 0]
    if sr == TGT_SR:
        return np.ascontiguousarray(x, dtype=np.float32)
    if resample_poly is None:
        raise RuntimeError("need scipy for resampling")
    from math import gcd
    g = gcd(int(sr), TGT_SR)
    return resample_poly(x, TGT_SR // g, int(sr) // g).astype(np.float32)


def fit_length(x: np.ndarray) -> list[np.ndarray]:
    """One clip -> whole 5 s rows.  Long clips split, short clips are padded."""
    if len(x) >= SEG_LEN:
        n = len(x) // SEG_LEN
        return list(x[: n * SEG_LEN].reshape(n, SEG_LEN))
    if len(x) >= int(MIN_KEEP_S * TGT_SR):
        return [np.pad(x, (0, SEG_LEN - len(x)))]
    return []


def info_any(fh):
    """sf.info on a stream that may not be seekable.

    A zip member handle is a forward-only stream: libsndfile's virtual IO wants
    seek/tell, so on a zip the direct call can fail.  Buffering the member is
    cheap (about 320 KB) and keeps one code path for every container.
    """
    try:
        return sf.info(fh)
    except Exception:
        return sf.info(io.BytesIO(fh.read()))


def read_any(fh) -> tuple[np.ndarray, int]:
    """sf.read with the same non-seekable fallback."""
    try:
        return sf.read(fh, dtype="float32", always_2d=True)
    except Exception:
        return sf.read(io.BytesIO(fh.read()), dtype="float32", always_2d=True)


# --------------------------------------------------------------------------- #
# audio sources: one interface over zip / loose wav / gzip-split tar
# --------------------------------------------------------------------------- #
ARCHIVE_SUFFIXES = (".zip", ".tar", ".tgz", ".tar.gz", ".tar.bz2", ".tar.xz")
CHUNK_RE = re.compile(r"^(?P<base>.+?)(?P<idx>\d{2})$")

# Compression magics.  Needed because splitting detection is name-based and
# Oceanship file names END IN A TWO-DIGIT AIS CODE: on a folder of loose wavs,
# "..._typecargo_70" and "..._typecargo_79" share a base and look exactly like a
# split set.  Requiring a real archive header on the first part kills that
# false positive, which otherwise sends the whole run down the streaming path
# and fails inside tarfile.
_MAGICS = (b"\x1f\x8b", b"PK\x03\x04", b"PK\x05\x06", b"7z\xbc\xaf\x27\x1c",
           b"BZh", b"\xfd7zXZ")


def _looks_like_archive(path: Path) -> bool:
    try:
        with open(path, "rb") as fh:
            head = fh.read(8)
    except OSError:
        return False
    return any(head.startswith(m) for m in _MAGICS)


def is_gzip(path: Path) -> bool:
    try:
        with open(path, "rb") as fh:
            return fh.read(2) == b"\x1f\x8b"
    except OSError:
        return False


def open_text(path: Path):
    """Read a text file that may or may not be gzipped, whatever it is named.

    Kaggle extracts archives when it ingests a dataset: a folder uploaded as
    `uwatr-workspace.zip` arrives one level deeper, `ShipsEar.zip` arrives as
    `shipsear/ShipsEar/`, and `oceanship_inventory.csv.gz` can arrive as a plain
    `oceanship_inventory.csv`.  Sniff the magic bytes rather than trusting the
    name, so every combination works.
    """
    if is_gzip(path):
        return gzip.open(path, "rt", newline="", encoding="utf-8")
    return open(path, "rt", newline="", encoding="utf-8")


def find_rel(workspace: Path, rel: str) -> Path | None:
    """Locate `rel` in a workspace, tolerating the un-archived form.

    Tries the shipped name, then the name Kaggle's extractor produces, then any
    file with the same leading stem in the same folder.
    """
    names = [rel]
    if rel.endswith(".gz"):
        names.append(rel[:-3])
    elif rel.endswith((".zip", ".tar")):
        names.append(str(Path(rel).with_suffix("")))
    for name in names:
        p = workspace / name
        if p.exists():
            return p
    for name in names:
        parent, base = workspace / Path(name).parent, Path(name).name.split(".")[0]
        if not parent.is_dir():
            continue
        for hit in sorted(parent.glob(base + "*")):
            if hit.exists():
                return hit
    return None


def copy_as(src: Path, dst: Path) -> None:
    """Copy into the published artifact under the documented name.

    If the artifact documents `x.csv.gz` and the input arrived already
    expanded, re-compress on the way in.  Downstream notebooks then always see
    the one documented spelling, whatever Kaggle did to the upload.
    """
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.name.endswith(".gz") and not is_gzip(src):
        with open(src, "rb") as fi, gzip.open(dst, "wb") as fo:
            shutil.copyfileobj(fi, fo)
    else:
        shutil.copy2(src, dst)


class MultiPartReader(io.RawIOBase):
    """Several split chunks as one read-only stream.

    Only concatenation happens here; tarfile does the gunzip on top, so a
    truncated set surfaces as a clean gzip error rather than silent garbage.
    """

    def __init__(self, parts: list[Path]) -> None:
        super().__init__()
        self._parts = list(parts)
        self._fh = None
        self.bytes_read = 0

    def readable(self) -> bool:
        return True

    def readinto(self, buf) -> int:  # type: ignore[override]
        while True:
            if self._fh is None:
                if not self._parts:
                    return 0
                self._fh = open(self._parts.pop(0), "rb")
            n = self._fh.readinto(buf)
            if n:
                self.bytes_read += n
                return n
            self._fh.close()
            self._fh = None


class Source:
    """Base: records() gives the canonical order, open_audio() gives bytes."""

    label = "source"
    streamable = False

    def records(self) -> list[dict]:
        raise NotImplementedError

    def open_audio(self, rec: dict):
        raise NotImplementedError


class ZipSource(Source):
    """A .zip holding every clip.  Random access, and the container is cached.

    Reopening the ZipFile per clip is the single most expensive mistake available
    here: the official archive has 53,770 members, so its central directory is
    about 6 MB and re-parsing it per clip measured out at 3.9 clips/s against
    100 clips/s with one cached handle.

    key="stem" is right when clip identity is the file name somewhere else
    (Oceanship).  key="member" is right when the path itself carries the label
    (ShipsEar, whose stems repeat across groups).
    """

    def __init__(self, path: Path, meta: dict[str, dict], stem_re: re.Pattern | None,
                 key: str = "stem"):
        self.path = path
        self.meta = meta
        self.key = key
        self._zf: zipfile.ZipFile | None = None
        self._members: dict[str, str] = {}
        with zipfile.ZipFile(path) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                stem = Path(info.filename).stem
                if stem_re is not None and not stem_re.match(stem):
                    continue
                k = info.filename if key == "member" else stem
                self._members.setdefault(k, info.filename)

    def _zip(self) -> zipfile.ZipFile:
        if self._zf is None:
            self._zf = zipfile.ZipFile(self.path)
        return self._zf

    def records(self) -> list[dict]:
        rows = []
        for k, member in self._members.items():
            m = self.meta.get(k)
            if m is not None:
                rows.append({"key": k, "member": member, **m})
        return rows

    def open_audio(self, rec: dict):
        return self._zip().open(rec["member"])

    def close(self) -> None:
        if self._zf is not None:
            self._zf.close()
            self._zf = None


class DirSource(Source):
    """Loose wav files on disk (the extracted form of a release).

    `files` maps the key used by the record table straight to a path, which is
    what DeepShip needs: its identity is the dataset-relative path, not the stem.
    """

    def __init__(self, root: Path, meta: dict[str, dict], stem_re: re.Pattern | None,
                 files: dict[str, Path] | None = None):
        self.root = root
        self.meta = meta
        self._files: dict[str, Path] = dict(files) if files is not None else {}
        if files is None:
            for p in sorted(root.rglob("*")):
                if p.is_file() and p.suffix.lower() in (".wav", ".flac"):
                    stem = p.stem
                    if stem_re is not None and not stem_re.match(stem):
                        continue
                    self._files.setdefault(stem, p)

    def records(self) -> list[dict]:
        rows = []
        for k, path in self._files.items():
            m = self.meta.get(k)
            if m is not None:
                rows.append({"key": k, "path": str(path), **m})
        return rows

    def open_audio(self, rec: dict):
        return open(rec["path"], "rb")


class StreamSource(Source):
    """A gzip-compressed tar split into byte chunks: one sequential pass only.

    Random access is impossible here, so this source is used when no .zip is
    present.  It is why measure() and write() are separable: this one only ever
    gets the write pass, with a generous allocation.
    """

    streamable = True

    def __init__(self, groups: list[tuple[str, list[Path]]], meta: dict[str, dict],
                 stem_re: re.Pattern | None):
        self.groups = groups
        self.meta = meta
        self.stem_re = stem_re

    def records(self) -> list[dict]:
        return [{"key": k, **v} for k, v in self.meta.items()]

    def iter_members(self):
        for base, parts in self.groups:
            reader = MultiPartReader(parts)
            with tarfile.open(fileobj=reader, mode="r|*") as tf:
                for member in tf:
                    if not member.isfile():
                        continue
                    stem = Path(member.name).stem
                    if self.stem_re is not None and not self.stem_re.match(stem):
                        continue
                    fh = tf.extractfile(member)
                    if fh is not None:
                        yield stem, fh


def resolve_source(root: Path, meta: dict[str, dict], stem_re: re.Pattern | None) -> Source:
    """Prefer random access, fall back to streaming.

    Accepts the folder that holds the audio or the archive itself, since a
    dataset can be mounted either way and both are natural to pass in.

    Everything is decided from ONE walk.  The obvious implementation - look for
    a zip, then look for split parts, then list the wavs - walks the tree three
    times, and when the release is the extracted form that is three passes over
    53,770 files on a mounted dataset.
    """
    if root.is_file():
        if root.name.lower().endswith(".zip"):
            return ZipSource(root, meta, stem_re)
        if _looks_like_archive(root):
            return StreamSource([(root.name, [root])], meta, stem_re)
        root = root.parent

    zips: list[Path] = []
    loose: dict[str, Path] = {}
    parts: list[Path] = []
    # os.walk rather than Path.rglob: rglob builds a Path object per entry and
    # stats every one to decide if it is a directory.  On the extracted release
    # that is 53,770 stats plus 53,770 objects, which measured in the minutes on
    # a mounted dataset.  os.walk hands back names that are already known to be
    # files, so there is nothing to stat and nothing to allocate for the ones we
    # skip.
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        base = Path(dirpath)
        for name in sorted(filenames):
            low = name.lower()
            if low.endswith(".zip"):
                zips.append(base / name)
            elif low.endswith(".wav") or low.endswith(".flac"):
                stem = name.rsplit(".", 1)[0]
                if stem_re is None or stem_re.match(stem):
                    loose.setdefault(stem, base / name)
            elif CHUNK_RE.match(name):
                parts.append(base / name)

    if zips:
        zips.sort(key=lambda p: -p.stat().st_size)
        return ZipSource(zips[0], meta, stem_re)

    groups: dict[tuple, list[tuple[int, Path]]] = {}
    for p in parts:
        m = CHUNK_RE.match(p.name)
        if m:
            groups.setdefault((p.parent, m.group("base")), []).append((int(m.group("idx")), p))
    sets = []
    for (_, base), items in groups.items():
        if len(items) < 2:
            continue
        items.sort(key=lambda t: t[0])
        first = items[0][1]
        if _looks_like_archive(first):
            sets.append((base, [q for _, q in items]))
    if sets:
        return StreamSource(sets, meta, stem_re)

    return DirSource(root, meta, stem_re, files=loose)


# --------------------------------------------------------------------------- #
# per-corpus record tables
# --------------------------------------------------------------------------- #
OCEANSHIP_STEM = re.compile(
    r"^(?P<stem>[0-9T:.Z-]+_\d+_id_[^_]+_type[a-zA-Z]+_\d+(?:_\d+)?)$")


def oceanship_meta(inventory: Path) -> dict[str, dict]:
    """Oceanship: identity and grouping come from the official CSV-derived inventory."""
    out: dict[str, dict] = {}
    with open_text(inventory) as fh:
        for r in csv.DictReader(fh):
            out[r["clip_id"]] = {
                "clip_id": r["clip_id"],
                "rec_key": r["rec_key"],
                "label": r["label"],
                "label_idx": r["label_idx"],
                "mmsi": r["mmsi"],
                "official_split": r["official_split"],
                "ais_type_code": r["ais_type_code"],
            }
    return out


def deepship_meta(recordings: Path) -> dict[str, dict]:
    """DeepShip: identity comes from the reconstructed recording table.

    wav_rel is relative to the dataset root, e.g. Cargo/20171104-1/1.wav.
    """
    out: dict[str, dict] = {}
    with recordings.open(encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            rel = r["wav_rel"].replace("\\", "/")
            out[rel] = {
                "clip_id": rel,
                "rec_key": r["rec_key"],
                "class_dir": r["class_dir"],
                "date": r["date"],
                "ship_name": r.get("ship_name", ""),
                "join_method": r.get("join_method", ""),
            }
    return out


SHIPSEAR_MEMBER = re.compile(r"^[^/]+/(?P<class_dir>[^/]+)/(?P<group>[^/]+)/(?P<stem>[^/]+)$")

# `shipsear_5s_16k/<class_dir>/<group>/<file>.wav` is the shape inside the zip;
# matching on the last four components means the extracted tree and the archive
# produce the same clip_id.
SHIPSEAR_TAIL = 4


def shipsear_meta(zip_path: Path) -> dict[str, dict]:
    """ShipsEar: the slice keeps class and group in the path, nothing else.

    The shipped *_list*.txt files are not used: their train/test split puts the
    same vessel on both sides (measured: 445/445 test clips contaminated in all
    six splits), so it is a leakage demo, not a protocol.
    """
    out: dict[str, dict] = {}
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            if info.is_dir() or not info.filename.lower().endswith(".wav"):
                continue
            m = SHIPSEAR_MEMBER.match(info.filename.replace("\\", "/"))
            if not m:
                continue
            out[info.filename] = {
                "clip_id": info.filename,
                "member": info.filename,
                "class_dir": m.group("class_dir"),
                "group": m.group("group"),
            }
    return out


def shipsear_meta_dir(root: Path) -> tuple[dict[str, dict], dict[str, Path]]:
    """The same records, read from the tree Kaggle extracted the zip into.

    Kaggle expands archives when a dataset is ingested, so the upload arrives as
    `shipsear/ShipsEar/shipsear_5s_16k/<class_dir>/<group>/<file>.wav`.  Keying
    on the last four path components reproduces the zip member name exactly, so
    `index.csv` is identical either way and a later switch cannot silently
    renumber the corpus.
    """
    out: dict[str, dict] = {}
    files: dict[str, Path] = {}
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix.lower() != ".wav":
            continue
        parts = p.relative_to(root).parts
        if len(parts) < SHIPSEAR_TAIL:
            continue
        key = "/".join(parts[-SHIPSEAR_TAIL:])
        m = SHIPSEAR_MEMBER.match(key)
        if not m:
            continue
        out.setdefault(key, {
            "clip_id": key,
            "member": key,
            "class_dir": m.group("class_dir"),
            "group": m.group("group"),
        })
        files.setdefault(key, p)
    return out, files


# --------------------------------------------------------------------------- #
# the two passes
# --------------------------------------------------------------------------- #
def _name_of(rec: dict) -> str:
    return str(rec.get("clip_id") or rec.get("key") or "?")


def _plan_row(rec: dict, info) -> dict:
    """One measured clip: how many 5 s rows it needs, and its real properties."""
    dur = info.frames / float(info.samplerate)
    n = len(fit_length(np.zeros(int(dur * TGT_SR) + 1, dtype=np.float32)))
    return {"rec": rec, "n_rows": n, "src_sr": info.samplerate,
            "src_ch": info.channels, "src_dur_s": round(dur, 3)}


def measure(src: Source, records: list[dict], limit: int) -> tuple[list[dict], list[dict]]:
    """Read headers only.  Returns (plan, failures).

    This is what makes the allocation exact, and what produces the duration
    distribution the data-description section needs.  Guessing instead would mean
    writing an 8 GB memmap and then rewriting it just to trim, and the trim copy
    does not fit alongside the array it is trimming.
    """
    if limit:
        records = records[:limit]
    plan: list[dict] = []
    failed: list[tuple[str, str]] = []
    t0 = time.time()
    for i, rec in enumerate(records, 1):
        try:
            fh = src.open_audio(rec)
            try:
                info = info_any(fh)
            finally:
                fh.close()
        except Exception as exc:
            failed.append((_name_of(rec), f"{type(exc).__name__}: {exc}"))
            continue
        plan.append(_plan_row(rec, info))
        if i % 5000 == 0:
            print(f"    measured {i:,}/{len(records):,}  {time.time()-t0:.0f}s", flush=True)
    print(f"  [measure] {len(plan):,} clips -> {sum(p['n_rows'] for p in plan):,} rows "
          f"in {time.time()-t0:.0f}s, {len(failed)} unreadable", flush=True)
    return plan, failed


def measure_stream(src: "StreamSource", records: list[dict], limit: int):
    """The same plan for an archive that can only be read front to back.

    Costs one extra decompression pass, roughly a few minutes, and buys an exact
    allocation.  Plan order is archive order and the write pass walks the archive
    the same way, so the two stay aligned row for row.
    """
    by_key = {r["key"]: r for r in records}
    plan: list[dict] = []
    failed: list[tuple[str, str]] = []
    t0 = time.time()
    for key, fh in src.iter_members():
        rec = by_key.get(key)
        if rec is None:
            continue
        try:
            info = info_any(fh)
        except Exception as exc:
            failed.append((key, f"{type(exc).__name__}: {exc}"))
            continue
        plan.append(_plan_row(rec, info))
        n = len(plan)
        if n % 5000 == 0:
            print(f"    measured {n:,}  {time.time()-t0:.0f}s", flush=True)
        if limit and n >= limit:
            break
    print(f"  [measure] {len(plan):,} clips -> {sum(p['n_rows'] for p in plan):,} rows "
          f"in {time.time()-t0:.0f}s, {len(failed)} unreadable", flush=True)
    return plan, failed


def write(job: str, src: Source, plan: list[dict], out: Path, reps: str,
          device: str, failed: list[tuple[str, str]],
          stream=None) -> dict:
    """The transform pass.  One memmap per representation, row aligned.

    `stream` is a zero-argument callable returning an iterator of (item, handle).
    A random-access source opens by record; a streamed archive re-walks itself in
    the order the plan was built in.

    A clip that measured fine but fails to decode here has its rows filled with
    the dB floor instead of shifting every later row.  Row alignment is the
    contract the splits and the metadata joins rest on, and every fill is counted
    and reported, so this is loud rather than silent.
    """
    want_mel = "mel" in reps
    want_spec = "spec" in reps
    n_rows = sum(p["n_rows"] for p in plan)
    shape = (n_rows, N_MELS, N_FRAMES)

    out.mkdir(parents=True, exist_ok=True)
    mel_fb, lin_fb = build_filterbanks()

    logmel = (np.lib.format.open_memmap(out / "logmel.npy", mode="w+",
                                        dtype=np.float16, shape=shape) if want_mel else None)
    spec = (np.lib.format.open_memmap(out / "spec.npy", mode="w+",
                                      dtype=np.float16, shape=shape) if want_spec else None)

    idx_fh = (out / "index.csv").open("w", newline="", encoding="utf-8")
    idx_w = csv.writer(idx_fh)
    idx_w.writerow(IDX_COLUMNS[job])

    floor = np.full((N_MELS, N_FRAMES), -TOP_DB, dtype=np.float16)
    row = 0
    filled = 0
    skipped_short = 0
    done = 0
    t0 = time.time()

    def fill_floor(item: dict) -> None:
        nonlocal row, filled
        k = item["n_rows"]
        if logmel is not None:
            logmel[row: row + k] = floor
        if spec is not None:
            spec[row: row + k] = floor
        for j in range(k):
            idx_w.writerow(idx_row(job, item["rec"], item, j, row + j))
        row += k
        filled += k

    def by_record():
        for item in plan:
            yield item, src.open_audio(item["rec"])

    def by_archive():
        want = {p["rec"]["key"]: p for p in plan}
        for key, fh in src.iter_members():
            item = want.get(key)
            if item is not None:
                yield item, fh

    source = stream() if stream is not None else by_record()
    consumed: set[int] = set()

    for i, (item, fh) in enumerate(source, 1):
        rec = item["rec"]
        try:
            x, sr = read_any(fh)
        except Exception as exc:
            failed.append((_name_of(rec), f"{type(exc).__name__}: {exc}"))
            x = None
        finally:
            try:
                fh.close()
            except Exception:
                pass
        consumed.add(id(item))

        if x is None:
            fill_floor(item)
            done += 1
            continue

        chunks = fit_length(to_mono_16k(x, sr))
        if not chunks:
            skipped_short += 1
            done += 1
            continue
        block = np.stack(chunks)

        for b0 in range(0, len(block), BATCH):
            sub = block[b0: b0 + BATCH]
            power = stft_power_batch(sub, device)
            if logmel is not None:
                logmel[row: row + len(sub)] = to_db(mel_fb @ power).astype(np.float16)
            if spec is not None:
                spec[row: row + len(sub)] = to_db(lin_fb @ power).astype(np.float16)
            for j in range(len(sub)):
                idx_w.writerow(idx_row(job, rec, item, b0 + j, row + j))
            row += len(sub)
        done += 1

        if done % 2000 == 0 or done == len(plan):
            el = time.time() - t0
            print(f"    [{done:,}/{len(plan):,}] rows={row:,}  {done/el:.1f} clip/s  "
                  f"eta {(len(plan)-done)/max(done/el,1e-9)/60:.1f} min", flush=True)

    # anything the plan expected but the walk never produced
    for item in plan:
        if id(item) not in consumed:
            failed.append((_name_of(item["rec"]), "not present in the archive"))
            fill_floor(item)

    for arr in (logmel, spec):
        if arr is not None:
            arr.flush()
    idx_fh.close()
    if row != n_rows:  # can only happen if the plan changed mid-run
        raise RuntimeError(f"row count drifted: wrote {row:,}, planned {n_rows:,}")
    return {"rows": row, "planned": n_rows, "skipped_short": skipped_short,
            "floor_filled": filled, "failed": failed,
            "elapsed_min": (time.time() - t0) / 60}


IDX_COLUMNS = {
    "oceanship": ["row", "clip_id", "clip_idx", "rec_key", "label", "label_idx",
                  "mmsi", "official_split", "ais_type_code", "src_sr", "src_ch", "src_dur_s"],
    "deepship": ["row", "clip_id", "clip_idx", "rec_key", "class_dir", "date",
                 "ship_name", "join_method", "src_sr", "src_ch", "src_dur_s"],
    "shipsear": ["row", "clip_id", "clip_idx", "class_dir", "group",
                 "src_sr", "src_ch", "src_dur_s"],
}


def idx_row(job: str, rec: dict, item: dict, clip_idx: int, row: int) -> list:
    base = [row, rec.get("clip_id", rec.get("key")), clip_idx]
    tail = [item["src_sr"], item["src_ch"], item["src_dur_s"]]
    if job == "oceanship":
        return base + [rec["rec_key"], rec["label"], rec["label_idx"], rec["mmsi"],
                       rec["official_split"], rec["ais_type_code"]] + tail
    if job == "deepship":
        return base + [rec["rec_key"], rec["class_dir"], rec["date"],
                       rec.get("ship_name", ""), rec.get("join_method", "")] + tail
    return base + [rec["class_dir"], rec["group"]] + tail


# --------------------------------------------------------------------------- #
# job drivers
#
# Measuring and writing are separate steps on purpose.  Measuring costs a full
# header pass over every clip, so it happens ONCE and its plan is handed to the
# write step.  It is also the only way the space guard can mean anything: the
# total is knowable before the first byte is written, and nowhere else.
#
# v4 fix: the guard used to be evaluated after the jobs ran, i.e. after ~14 GB
# had already been written, so it could only ever fire when it was too late to
# matter - it aborted on a full disk and left the artifact with no manifest.
# --------------------------------------------------------------------------- #
def prepare_job(job: str, src: Source, limit: int) -> dict:
    """Measure one job.  Writes nothing, returns the plan to hand to commit_job."""
    print()
    print("=" * 88)
    print(f"job: {job}   ({src.label})")
    print("=" * 88)
    records = src.records()
    print(f"  records available: {len(records):,}", flush=True)
    if not records:
        print("  NOTHING TO DO - the source was found but holds no matching clips.")
        return {"job": job, "src": src, "plan": [], "failed": [], "stream": None,
                "planned": 0, "error": "no records"}

    if getattr(src, "streamable", False):
        plan, failed = measure_stream(src, records, limit)
        stream = lambda: _archive_items(src, plan)  # noqa: E731
    else:
        plan, failed = measure(src, records, limit)
        stream = None

    if not plan:
        print("  NOTHING TO DO - no readable headers.")
        return {"job": job, "src": src, "plan": [], "failed": failed, "stream": None,
                "planned": 0, "error": "no readable audio"}

    report_plan(job, plan)
    return {"job": job, "src": src, "plan": plan, "failed": failed, "stream": stream,
            "planned": sum(p["n_rows"] for p in plan)}


def commit_job(prep: dict, out: Path, reps: str, device: str) -> dict:
    """Write one already-measured job and report it."""
    job, plan = prep["job"], prep["plan"]
    if not plan:
        return {"rows": 0, "planned": 0, "failed": prep["failed"],
                "error": prep.get("error", "nothing to do")}

    stats = write(job, prep["src"], plan, out, reps, device, prep["failed"],
                  stream=prep["stream"])

    lines = [
        "=" * 78,
        f"{job} feature cache",
        "=" * 78,
        f"clips planned   : {len(plan):,}",
        f"rows written    : {stats['rows']:,}  (planned {stats['planned']:,})",
        f"shape           : ({stats['rows']}, {N_MELS}, {N_FRAMES}) fp16 x {len(reps.split('+'))}",
        f"representations : {reps}",
        f"dropped short   : {stats['skipped_short']:,} clips under {MIN_KEEP_S:g}s",
        f"floor-filled    : {stats.get('floor_filled', 0):,} rows (unreadable audio)",
        f"failed          : {len(stats['failed'])}",
    ]
    lines += [f"    {k}: {v}" for k, v in stats["failed"][:10]]
    lines += [f"elapsed         : {stats['elapsed_min']:.1f} min", f"written         : {out}"]
    text = "\n".join(lines)
    print()
    print(text)
    (out / f"w2_build_{job}_report.txt").write_text(text, encoding="utf-8")
    return stats


def _archive_items(src: Source, plan: list[dict]):
    """Walk a streamed archive, yielding only the plan entries, in archive order."""
    want = {p["rec"]["key"]: p for p in plan}
    for key, fh in src.iter_members():
        item = want.get(key)
        if item is not None:
            yield item, fh


def report_plan(job: str, plan: list[dict]) -> None:
    """Print the class row counts and the duration distribution per class."""
    by_class: dict[str, list[float]] = collections.defaultdict(list)
    n_rows: dict[str, int] = collections.defaultdict(int)
    key = {"oceanship": "label", "deepship": "class_dir", "shipsear": "class_dir"}[job]
    for p in plan:
        c = str(p["rec"].get(key) or p["rec"].get("group", "?"))
        by_class[c].append(p["src_dur_s"])
        n_rows[c] += p["n_rows"]
    print(f"  {'class':<28}{'clips':>8}{'rows':>8}{'med dur s':>11}{'min':>7}{'max':>7}")
    for c in sorted(by_class, key=lambda x: -len(by_class[x])):
        d = np.array(by_class[c])
        print(f"  {c:<28}{len(d):>8,}{n_rows[c]:>8,}{np.median(d):>11.2f}{d.min():>7.2f}{d.max():>7.2f}")
    d = np.array([p["src_dur_s"] for p in plan])
    print(f"  {'(all)':<28}{len(d):>8,}{sum(n_rows.values()):>8,}"
          f"{np.median(d):>11.2f}{d.min():>7.2f}{d.max():>7.2f}")
    print(f"  classes: {len(by_class)}   short (<{MIN_KEEP_S:g}s, dropped): {(d < MIN_KEEP_S).sum():,}")
    if job == "shipsear":
        g = collections.Counter(str(p["rec"].get("group", "?")) for p in plan)
        print(f"  groups ({len(g)}): {dict(sorted(g.items()))}")
    sr = collections.Counter(p["src_sr"] for p in plan)
    ch = collections.Counter(p["src_ch"] for p in plan)
    print(f"  source sample rates: {dict(sr.most_common(4))}")
    print(f"  source channels    : {dict(ch.most_common(4))}")


def estimate_gb(job: str, n_rows: int, reps: str) -> float:
    return n_rows * BYTES_PER_ROW * len(reps.split("+")) / 1e9


# --------------------------------------------------------------------------- #
# assembly
# --------------------------------------------------------------------------- #
README = """# uwatr-features

Frozen acoustic representations for three underwater acoustic corpora, built by
`build_features.py` (also in this folder) so that every later experiment reuses
one feature space instead of re-deriving it.

## Layout

    build_features.py        the exact script that produced this
    manifest.json            counts, shapes, per-split row counts, config, hashes
    oceanship/{logmel.npy,spec.npy,index.csv}
    deepship/{logmel.npy,spec.npy,index.csv}
    shipsear/{logmel.npy,spec.npy,index.csv}
    splits/                  frozen protocol splits + leakage audit CSVs

## Representation (identical for all three corpora)

    16 kHz mono, 5 s rows, 80,000 samples
    log-mel  (N, 128, 313) fp16, 20-8000 Hz, Slaney-style area normalisation
    linear   (N, 128, 313) fp16, 513 bins average-pooled into 128 bands
    both in dB relative to the clip peak, floored at -80 dB

Short clips are padded, not dropped, and the *measured* source duration of every
row is in `index.csv` as `src_dur_s`.  That column exists so a
with/without-short-clips ablation is one join away: duration is class-correlated
(Oceanship Passenger median 3.67 s against Tug 5.00 s), so padding without
recording it would hand the model a silent shortcut.

## Joining a split

    oceanship  index.clip_id                  -> splits/oceanship_splits.csv.gz  (clip_id)
    deepship   index.(rec_key, clip_idx)      -> splits/deepship_splits.csv.gz   (rec_key, clip_idx)

`splits/` carries three protocols x five seeds each: `clip_random`,
`rec_random`, `vessel_disjoint`, plus the per-protocol leakage summary.  Use
`vessel_disjoint` for anything that claims to measure generalisation.

## Mounting

    /kaggle/input/<this-dataset>/oceanship/logmel.npy     np.load(..., mmap_mode="r")
    /kaggle/input/<this-dataset>/oceanship/index.csv
    /kaggle/input/<this-dataset>/splits/oceanship_splits.csv.gz

## Whatever the uploader did to the archive

Kaggle expands archives when it ingests a dataset.  A folder uploaded as
`uwatr-workspace.zip` arrives one level deeper, `ShipsEar.zip` arrives as
`shipsear/ShipsEar/shipsear_5s_16k/<class>/<group>/<file>.wav`, and
`oceanship_inventory.csv.gz` can arrive as a plain `.csv`.  Every input path here
accepts either form, the extracted ShipsEar tree produces the *same* `clip_id`s
as the archive (both match on the last four path components), and the published
artifact always uses the documented name and compression so nothing downstream
has to guess.
"""


def assemble(out: Path, workspace: Path | None, summaries: dict) -> None:
    """Copy the frozen splits in, then write manifest.json and README.md."""
    (out / "splits").mkdir(parents=True, exist_ok=True)
    copied, absent = [], []
    if workspace is not None:
        # (shipped name, name in the artifact).  The artifact always uses the
        # shipped spelling: Kaggle may have expanded the upload, and downstream
        # notebooks should not have to guess.
        wanted = (("splits/oceanship_splits.csv.gz", "oceanship_splits.csv.gz"),
                  ("splits/oceanship_leakage_summary.csv", "oceanship_leakage_summary.csv"),
                  ("splits/splits.csv.gz", "deepship_splits.csv.gz"),
                  ("splits/split_summary.csv", "deepship_split_summary.csv"),
                  ("metadata/oceanship_inventory.csv.gz", "oceanship_inventory.csv.gz"))
        for rel, name in wanted:
            src = find_rel(workspace, rel)
            if src is None:
                absent.append(rel)
                continue
            copy_as(src, out / "splits" / name)
            copied.append(name)
    if not copied:
        print("[assemble] WARNING: no workspace dataset found; splits/ will be empty")
    elif absent:
        print(f"[assemble] WARNING: not found in the workspace: {absent}")

    manifest = {"config": {"tgt_sr": TGT_SR, "seg_s": SEG_S, "n_fft": N_FFT, "hop": HOP,
                           "n_mels": N_MELS, "n_frames": N_FRAMES, "top_db": TOP_DB,
                           "min_keep_s": MIN_KEEP_S},
                "jobs": {}, "splits_copied": copied}
    for job, s in summaries.items():
        jd = out / job
        entry = {"rows": s.get("rows", 0), "planned": s.get("planned"),
                 "skipped_short": s.get("skipped_short", 0),
                 "n_failed": len(s.get("failed", [])),
                 "elapsed_min": round(s.get("elapsed_min", 0.0), 1),
                 "arrays": {}}
        for name in ("logmel.npy", "spec.npy"):
            p = jd / name
            if p.exists():
                a = np.load(p, mmap_mode="r")
                entry["arrays"][name] = {"shape": list(a.shape), "dtype": str(a.dtype),
                                         "gb": round(p.stat().st_size / 1e9, 3)}
                del a
        idx = jd / "index.csv"
        if idx.exists():
            with idx.open(encoding="utf-8") as fh:
                rows = list(csv.DictReader(fh))
            entry["index_rows"] = len(rows)
            if job == "oceanship" and rows:
                entry["classes"] = len({r["label"] for r in rows})
                entry["vessels"] = len({r["mmsi"] for r in rows})
            elif job == "deepship" and rows:
                entry["classes"] = len({r["class_dir"] for r in rows})
            elif job == "shipsear" and rows:
                entry["classes"] = len({r["class_dir"] for r in rows})
        manifest["jobs"][job] = entry

    (out / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (out / "README.md").write_text(README, encoding="utf-8")

    print()
    print("=" * 88)
    print("output")
    print("=" * 88)
    total = 0
    for p in sorted(out.rglob("*")):
        if p.is_file():
            total += p.stat().st_size
            print(f"  {p.stat().st_size:>15,}  {p.relative_to(out)}")
    print(f"  {'-' * 15}")
    print(f"  {total:>15,} total = {total/1e9:.2f} GB")


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--job", default="all",
                    choices=["oceanship", "deepship", "shipsear", "all"])
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--workspace", type=Path, default=None,
                    help="mounted uwatr-workspace dataset (for splits + inventory)")
    ap.add_argument("--oceanship-audio", type=Path, default=None)
    ap.add_argument("--oceanship-inventory", type=Path, default=None,
                    help="clip_id/label/mmsi table; .csv.gz or an expanded .csv")
    ap.add_argument("--deepship-root", type=Path, default=None)
    ap.add_argument("--deepship-recordings", type=Path, default=None)
    ap.add_argument("--shipsear-zip", type=Path, default=None,
                    help="ShipsEar.zip, or the ShipsEar/ folder Kaggle extracted it into")
    ap.add_argument("--reps", default="mel+spec", choices=["mel", "spec", "mel+spec"])
    ap.add_argument("--device", default="cuda" if (torch and torch.cuda.is_available()) else "cpu")
    ap.add_argument("--limit", type=int, default=0, help="first N clips per job (smoke test)")
    ap.add_argument("--probe", action="store_true",
                    help="headers only: report counts and durations, write nothing")
    ap.add_argument("--skip-check", action="store_true", help="do not enforce the space guard")
    args = ap.parse_args()

    print("=" * 88)
    print("UATR frozen feature dataset")
    print("=" * 88)
    print(f"  out        : {args.out}")
    print(f"  reps       : {args.reps}   device: {args.device}")
    print(f"  torch      : {'yes ' + torch.__version__ if torch else 'NOT AVAILABLE (cpu only)'}")
    print(f"  resample   : {'scipy resample_poly' if resample_poly else 'MISSING - pip install scipy'}")

    jobs = ["oceanship", "deepship", "shipsear"] if args.job == "all" else [args.job]
    summaries: dict[str, dict] = {}

    ws = args.workspace
    inv = args.oceanship_inventory or (find_rel(ws, "metadata/oceanship_inventory.csv.gz")
                                       if ws else None)
    recs = args.deepship_recordings or (find_rel(ws, "deepship/deepship_recordings.csv")
                                        if ws else None)
    sea = args.shipsear_zip or (find_rel(ws, "shipsear/ShipsEar.zip") if ws else None)
    if sea is None and ws is not None:
        # extracted by Kaggle: shipsear/ShipsEar/ instead of shipsear/ShipsEar.zip
        for cand in sorted((ws / "shipsear").glob("ShipsEar*")) if (ws / "shipsear").is_dir() else []:
            if cand.is_dir():
                sea = cand
                break

    # ------------------------------------------------------------------ #
    # phase 1: resolve a source per job and MEASURE it.  Nothing is written.
    # ------------------------------------------------------------------ #
    prepared: list[dict] = []

    if "oceanship" in jobs:
        if args.oceanship_audio and inv and Path(inv).exists():
            meta = oceanship_meta(Path(inv))
            print(f"  oceanship  : {len(meta):,} inventory rows, audio root {args.oceanship_audio}")
            src = resolve_source(Path(args.oceanship_audio), meta, OCEANSHIP_STEM)
            src.label = f"{type(src).__name__} {getattr(src, 'path', getattr(src, 'root', ''))}"
            prepared.append(prepare_job("oceanship", src, args.limit))
        else:
            print("  oceanship  : SKIPPED (need --oceanship-audio and the inventory csv)")

    if "deepship" in jobs:
        if args.deepship_root and recs and Path(recs).exists():
            meta = deepship_meta(Path(recs))
            root = Path(args.deepship_root)
            # the dataset may be mounted with an extra folder level; find the one
            # that actually holds Cargo/ or Passenger/
            if not any((root / c).is_dir() for c in ("Cargo", "Passenger", "Tanker", "Tug")):
                for cand in sorted(root.rglob("*")):
                    if cand.is_dir() and (cand / "Cargo").is_dir():
                        root = cand
                        break
            print(f"  deepship   : {len(meta):,} recordings, resolved root {root}")
            present = {rel: (root / rel) for rel in meta if (root / rel).exists()}
            print(f"  deepship   : {len(present):,} of {len(meta):,} wav files present")
            src = DirSource(root, meta, None, files=present)
            src.label = f"DirSource {root}"
            prepared.append(prepare_job("deepship", src, args.limit))
        else:
            print("  deepship   : SKIPPED (need --deepship-root and the recordings csv)")

    if "shipsear" in jobs:
        if sea and Path(sea).exists():
            sea = Path(sea)
            if sea.is_dir():
                # Kaggle expanded the upload: read the extracted tree instead of
                # the archive, and it is faster because nothing is decompressed.
                meta, files = shipsear_meta_dir(sea)
                print(f"  shipsear   : {len(meta):,} clips in the extracted tree "
                      f"{sea.name}/  (no archive present)")
                src = DirSource(sea, meta, None, files=files)
            else:
                meta = shipsear_meta(sea)
                print(f"  shipsear   : {len(meta):,} clips in {sea.name}")
                src = ZipSource(sea, meta, None, key="member")
            src.label = f"{type(src).__name__} {sea.name}"
            prepared.append(prepare_job("shipsear", src, args.limit))
        else:
            print("  shipsear   : SKIPPED (need --shipsear-zip, a .zip or the "
                  "extracted ShipsEar/ folder)")

    if args.probe:
        print()
        print("  probe only - nothing written.  Re-run without --probe to build.")
        return 0

    # ------------------------------------------------------------------ #
    # phase 2: ONE space check for the whole run, before the first write
    # ------------------------------------------------------------------ #
    planned_rows = sum(p.get("planned", 0) for p in prepared)
    need = estimate_gb("all", planned_rows, args.reps)
    args.out.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(args.out).free / 1e9
    headroom = 1.0 if args.reps == "mel" else 2.0
    print()
    print(f"  planned rows {planned_rows:,}  ->  about {need:.2f} GB of arrays "
          f"for --reps {args.reps};  {free:.1f} GB free")
    if not args.skip_check and need > free - headroom:
        print(f"  ABORT BEFORE WRITING: needs {need:.2f} GB but only {free:.1f} GB is "
              f"free ({headroom:.0f} GB kept as headroom).")
        print("  options, in order of least loss:")
        print("    --reps mel          halves the cache (one representation, not two)")
        print("    free space on /kaggle/working, or delete a previous run's output")
        print("    --limit N           build a subset for a smoke test")
        print("  Nothing was written, so a re-run starts clean.")
        return 1

    # ------------------------------------------------------------------ #
    # phase 3: write, then assemble
    # ------------------------------------------------------------------ #
    # the artifact carries the script that produced it
    me = Path(__file__).resolve()
    if me != (args.out / "build_features.py").resolve():
        shutil.copy2(me, args.out / "build_features.py")
    else:
        print("  build_features.py: already at the output root (run from the output"
              " folder), nothing to copy")

    summaries: dict[str, dict] = {}
    for prep in prepared:
        summaries[prep["job"]] = commit_job(prep, args.out / prep["job"],
                                            args.reps, args.device)
        if hasattr(prep["src"], "close"):
            prep["src"].close()

    assemble(args.out, ws if (ws and ws.exists()) else None, summaries)

    print()
    print("  next: download this folder and upload it as a Kaggle Dataset, or")
    print("        create the dataset straight from this notebook version's output.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/scripts/w1_04_oceanship_inventory.py
"""W1-04  Oceanship inventory, leakage audit and split generation.

This script runs entirely on the official metadata CSVs published by the
dataset authors, i.e. **it needs no audio**.  That matters because the audio
(23.4 GiB train + 5.1 GiB test on 123pan) is the slow part to obtain, while
the CSVs are a 27 MB GitHub download.

What the official CSVs give us
------------------------------
`oceanship_fg_{train,test}.csv` have these columns:

    text_path, wav_path, label, x, y, sog, cog, true_heading,
    mmsi, id, ais_timestamp, label_idx

`wav_path` looks like

    ./3090latest_wav/{audio_timestamp}_{n}_id_{d}_type{cls}_{ais_code}[_{seg}].pt

Three distinct notions of "identity" are recoverable, and this script keeps
them separate on purpose, because the whole point of the paper is that
confusing them is what produces the inflated numbers in the literature:

    clip        one row                     (5 s of audio)
    recording   (mmsi, ais_timestamp)       one acoustic session of one ship
    vessel      mmsi                        the ship itself

The authors' own train/test CSVs are split at the *clip* level.  Everything
else here is us re-deriving a stricter protocol from the same metadata.

Outputs (into --out)
--------------------
    oceanship_inventory.csv          one row per clip, with derived keys
    oceanship_splits.csv.gz          our own 3 protocols x N seeds, clip level
    oceanship_leakage_summary.csv    leakage rate per protocol per level
    w1_04_oceanship_report.txt       human-readable report

Usage
-----
    python w1_04_oceanship_inventory.py \
        --csv-dir data/external/oceanship_gh \
        --out out

Nothing here talks to the network or reads audio.
"""

from __future__ import annotations

import argparse
import collections
import csv
import gzip
import hashlib
import random
import re
import sys
from pathlib import Path

# 70/15/15, same as the DeepShip pipeline so the two are directly comparable
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

PROTOCOLS = ("clip_random", "rec_random", "vessel_disjoint")

# wav_path -> ./3090latest_wav/{ts}_{n}_id_{d}_type{cls}_{ais}[_{seg}].pt
WAV_RE = re.compile(
    r"^(?P<ts>[0-9T:.Z-]+)_(?P<n>\d+)_id_(?P<dep>[^_]+)_type(?P<cls>[a-zA-Z]+)_(?P<ais>\d+)"
    r"(?:_(?P<seg>\d+))?$"
)


def stable_seed(protocol: str, index: int) -> int:
    """Deterministic per-(protocol, seed-index) seed.

    This must not use the builtin ``hash()``: string hashing is salted per
    process by PYTHONHASHSEED, so the same code on the same CSVs produced a
    different "frozen" split every run.  Measured before the fix: 374,085 of
    806,310 (protocol, seed, clip_id) rows were assigned differently between two
    runs of this script, while the leaked-recordings rate of the audited
    protocols moved with them (clip_random seed 0: 91.4% vs 91.8%).

    Uses the same construction as w1_02_make_splits.stable_seed so the
    Oceanship and DeepShip protocols are seeded identically.
    """
    digest = hashlib.sha256(f"{protocol}|{index}".encode()).digest()
    return int.from_bytes(digest[:8], "big")


def load_side(csv_dir: Path, side: str) -> list[dict]:
    """Read one official CSV and attach the derived keys we need."""
    path = csv_dir / f"oceanship_fg_{side}.csv"
    if not path.exists():
        raise SystemExit(f"missing {path}")
    rows: list[dict] = []
    with open(path, newline="", encoding="utf-8", errors="replace") as fh:
        for r in csv.DictReader(fh):
            stem = Path(r["wav_path"]).name
            m = WAV_RE.match(stem[:-3] if stem.endswith(".pt") else stem)
            if not m:
                # keep it, but flag it: an unparsable name is itself a finding
                rows.append({"raw": r, "parsed": None})
                continue
            rows.append({"raw": r, "parsed": m.groupdict()})
    return rows


def build_inventory(csv_dir: Path, out_dir: Path) -> tuple[list[dict], dict]:
    """One row per clip, plus a diagnostic dict about the metadata itself."""
    inv: list[dict] = []
    diag = collections.Counter()
    dep_mismatch: collections.defaultdict[str, collections.Counter] = collections.defaultdict(
        collections.Counter
    )

    for side in ("train", "test"):
        for item in load_side(csv_dir, side):
            r, p = item["raw"], item["parsed"]
            if p is None:
                diag["unparsable_wav_path"] += 1
                continue
            # the filename carries `id_{d}`; the CSV carries its own `id`
            if r["id"] and p["dep"] != r["id"]:
                dep_mismatch[r["id"]][p["dep"]] += 1
            inv.append(
                {
                    "official_split": side,
                    "clip_id": Path(r["wav_path"]).stem,
                    "label": r["label"],
                    "label_idx": r["label_idx"],
                    "mmsi": r["mmsi"],
                    "csv_id": r["id"],
                    "fname_id": p["dep"],
                    "ais_type_code": p["ais"],
                    "audio_ts": p["ts"],
                    "ais_timestamp": r["ais_timestamp"],
                    "sog": r["sog"],
                    "cog": r["cog"],
                    "true_heading": r["true_heading"],
                    "x": r["x"],
                    "y": r["y"],
                    # recording = one acoustic session of one ship
                    "rec_key": f"{r['mmsi']}|{r['ais_timestamp']}",
                }
            )

    all_dep = collections.Counter(v for c in dep_mismatch.values() for v in c)
    diag["csv_id_to_fname_id_mismatch"] = sum(sum(c.values()) for c in dep_mismatch.values())
    diag["distinct_csv_id"] = len(dep_mismatch) + sum(
        1 for r in inv if r["csv_id"] not in dep_mismatch
    )
    diag["distinct_fname_id"] = len(all_dep)

    with gzip.open(out_dir / "oceanship_inventory.csv.gz", "wt", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(inv[0].keys()))
        w.writeheader()
        w.writerows(inv)
    return inv, diag


# --------------------------------------------------------------------------- #
# leakage auditing
# --------------------------------------------------------------------------- #
def leakage_for(assign: dict[str, str], inv: list[dict]) -> dict:
    """Given clip_id -> split, report how badly each identity level leaks.

    For 'clip_random' the notion of a clip leaking is meaningless (every clip
    is its own unit), so the interesting number there is how much of the test
    set shares a recording / a vessel with train.
    """
    train_rec, train_ves, train_clip = set(), set(), set()
    for r in inv:
        if assign[r["clip_id"]] == "train":
            train_rec.add(r["rec_key"])
            train_ves.add(r["mmsi"])
            train_clip.add(r["clip_id"])

    test = [r for r in inv if assign[r["clip_id"]] == "test"]
    if not test:
        return {"n_test": 0}
    n = len(test)
    return {
        "n_test": n,
        "n_test_vessels": len({r["mmsi"] for r in test}),
        "n_test_recordings": len({r["rec_key"] for r in test}),
        "pct_test_share_vessel": round(100 * sum(r["mmsi"] in train_ves for r in test) / n, 1),
        "pct_test_share_recording": round(100 * sum(r["rec_key"] in train_rec for r in test) / n, 1),
        "pct_test_share_clip": round(100 * sum(r["clip_id"] in train_clip for r in test) / n, 1),
    }


def official_assignment(inv: list[dict]) -> dict[str, str]:
    return {r["clip_id"]: r["official_split"] for r in inv}


# --------------------------------------------------------------------------- #
# split generation
# --------------------------------------------------------------------------- #
def _stratified_partition(units: list[str], classes: dict, rng: random.Random) -> dict:
    """70/15/15 by class, at whatever unit granularity the caller chose.

    `units` is a list of identity keys (a clip id, a rec_key or an MMSI list
    element) and `classes` maps each key to its label.  Splitting on *clips* is
    what the literature does; splitting on recordings or vessels is what we do.
    Keeping one function for all three means the protocols differ only in their
    unit definition, never in their arithmetic - which is the whole point when
    the paper's claim is that the difference in reported accuracy comes from the
    unit, not from the model.
    """
    by_class: dict[str, list] = collections.defaultdict(list)
    for u in units:
        by_class[classes[u]].append(u)

    assign: dict[str, str] = {}
    for cls in sorted(by_class):
        items = by_class[cls]
        rng.shuffle(items)
        n = len(items)
        if n < 3:  # can't populate three splits; keep it honest and visible
            for u in items:
                assign[u] = "train"
            continue
        n_train = int(round(n * TRAIN_FRAC))
        n_val = int(round(n * VAL_FRAC))
        # never leave val or test empty
        n_train = min(n_train, n - 2)
        n_val = max(1, min(n_val, n - n_train - 1))
        for i, u in enumerate(items):
            assign[u] = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
    return assign


def build_splits(inv: list[dict], n_seeds: int, classes_kept: set[str] | None = None):
    """Return (clip-level split rows, per-protocol/seed leakage rows).

    `classes_kept` restricts the benchmark to classes that have enough vessels
    to populate a 3-way vessel-disjoint split.  Classes with one or two vessels
    cannot appear in train AND val AND test no matter how you slice them, so
    keeping them would silently produce classes with an empty test set - which
    is exactly the kind of quiet flaw this paper is about.
    """
    if classes_kept is not None:
        inv = [r for r in inv if r["label"] in classes_kept]

    clip_class = {r["clip_id"]: r["label"] for r in inv}
    rec_class: dict[str, str] = {}
    rec_clips: dict[str, list[str]] = collections.defaultdict(list)
    ves_clips: dict[str, list[str]] = collections.defaultdict(list)
    for r in inv:
        rec_class.setdefault(r["rec_key"], r["label"])
        rec_clips[r["rec_key"]].append(r["clip_id"])
        ves_clips[r["mmsi"]].append(r["clip_id"])

    split_rows: list[dict] = []
    leak_rows: list[dict] = []

    for protocol in PROTOCOLS:
        for seed_index in range(n_seeds):
            rng = random.Random(stable_seed(protocol, seed_index))

            if protocol == "clip_random":
                clip_assign = _stratified_partition(list(clip_class), clip_class, rng)
            elif protocol == "rec_random":
                unit_assign = _stratified_partition(list(rec_clips), rec_class, rng)
                clip_assign = {c: unit_assign[k] for k, v in rec_clips.items() for c in v}
            else:  # vessel_disjoint
                # a vessel can (rarely) appear under two labels; take the majority
                votes: dict[str, collections.Counter] = collections.defaultdict(collections.Counter)
                for r in inv:
                    votes[r["mmsi"]][r["label"]] += 1
                ves_class = {v: c.most_common(1)[0][0] for v, c in votes.items()}
                unit_assign = _stratified_partition(list(ves_clips), ves_class, rng)
                clip_assign = {c: unit_assign[v] for v, cl in ves_clips.items() for c in cl}

            for c, s in clip_assign.items():
                split_rows.append(
                    {"protocol": protocol, "seed": seed_index, "clip_id": c, "split": s}
                )

            leak = leakage_for(clip_assign, inv)
            leak["protocol"] = protocol
            leak["seed"] = seed_index
            leak_rows.append(leak)

    return split_rows, leak_rows


# --------------------------------------------------------------------------- #
# report
# --------------------------------------------------------------------------- #
def write_report(inv, diag, leak_rows, official_leak, out_dir: Path, args, dropped: dict) -> None:
    L: list[str] = []
    add = L.append

    add("=" * 88)
    add("Oceanship inventory and leakage audit")
    add("=" * 88)
    add("")
    add(f"source CSVs : {args.csv_dir}")
    add(f"clips       : {len(inv):,}")
    add(f"recordings  : {len({r['rec_key'] for r in inv}):,}   (= distinct (mmsi, ais_timestamp))")
    add(f"vessels     : {len({r['mmsi'] for r in inv}):,}   (= distinct MMSI)")
    add(f"classes     : {len({r['label'] for r in inv})}")
    add("")

    add("clips per split (as published by the authors)")
    add("-" * 88)
    by = collections.Counter(r["official_split"] for r in inv)
    for s in ("train", "val", "test"):
        if by.get(s):
            add(f"  {s:<6}{by[s]:>9,}")
    add("")

    add("vessels and clips per class (vessel counts decide what a split can do)")
    add("-" * 88)
    ves: dict[str, set] = collections.defaultdict(set)
    cl: collections.Counter = collections.Counter()
    for r in inv:
        ves[r["label"]].add(r["mmsi"])
        cl[r["label"]] += 1
    add(f"  {'class':<30}{'vessels':>9}{'clips':>9}{'usable':>9}")
    for c in sorted(ves, key=lambda x: -len(ves[x])):
        add(f"  {c:<30}{len(ves[c]):>9,}{cl[c]:>9,}{'yes' if len(ves[c]) >= args.min_vessels else 'NO':>9}")
    add("")
    if dropped:
        lost = sum(dropped.values())
        add(f"  dropped {len(dropped)} class(es) with < {args.min_vessels} vessels: "
            f"{', '.join(sorted(dropped))}")
        add(f"  cost: {lost:,} clips ({100 * lost / (len(inv) + lost):.2f}%) - these classes")
        add("  cannot appear in all three splits at vessel level, so keeping them would")
        add("  create classes with an empty test set.  Excluded explicitly, not silently.")
        add("")

    add("=" * 88)
    add("THE HEADLINE NUMBER: the authors' own split is clip-level")
    add("=" * 88)
    add("")
    add("  A clip-level split cannot keep a ship out of both sides, because one")
    add("  ship contributes hundreds of clips.  Measured on the official CSVs:")
    add("")
    add(f"  {'metric':<46}{'value':>12}")
    add(f"  {'test clips':<46}{official_leak['n_test']:>12,}")
    add(f"  {'test clips whose RECORDING is also in train':<46}{official_leak['pct_test_share_recording']:>11.1f}%")
    add(f"  {'test clips whose VESSEL is also in train':<46}{official_leak['pct_test_share_vessel']:>11.1f}%")
    add("")
    add("  -> the vessel column is the one that decides whether the model is")
    add("     measuring ship-type recognition or ship memorisation.")
    add("")

    add("=" * 88)
    add(f"our own protocols, {args.n_seeds} seeds each (70/15/15, stratified by class)")
    add("=" * 88)
    add("")
    add(f"  {'protocol':<17}{'mean test clips':>16}{'share recording':>17}{'share vessel':>14}")
    agg: dict[str, list[dict]] = collections.defaultdict(list)
    for r in leak_rows:
        agg[r["protocol"]].append(r)
    for p in PROTOCOLS:
        rs = agg[p]
        mean = lambda k: sum(x[k] for x in rs) / len(rs)  # noqa: E731
        add(f"  {p:<17}{mean('n_test'):>16,.0f}{mean('pct_test_share_recording'):>16.1f}%{mean('pct_test_share_vessel'):>13.1f}%")
    add("")
    add("  rec_random should read 0.0% at the recording level but NOT at the")
    add("  vessel level - that gap is the finding the paper is built on.")
    add("  vessel_disjoint must read 0.0% at both.")
    add("")

    if diag.get("csv_id_to_fname_id_mismatch"):
        add("=" * 88)
        add("metadata integrity notes")
        add("=" * 88)
        add("")
        add(f"  wav_path not matching the expected pattern : {diag.get('unparsable_wav_path', 0):,}")
        add(f"  rows where the CSV `id` differs from the `id_` in wav_path : {diag['csv_id_to_fname_id_mismatch']:,}")
        add("")
        add("  Both columns take values from the same small set, but they disagree, so")
        add("  at least one of them is not the deployment id it appears to be.  We do")
        add("  not rely on either: vessel identity comes from `mmsi`, which is a real")
        add("  9-digit MMSI and is 100% populated.  Recorded for the paper's data")
        add("  description section only.")
        add("")

    text = "\n".join(L) + "\n"
    (out_dir / "w1_04_oceanship_report.txt").write_text(text, encoding="utf-8")
    print(text)


def main() -> int:
    ap = argparse.ArgumentParser(description="Oceanship inventory, leakage audit, splits.")
    root = Path(__file__).resolve().parent.parent
    ap.add_argument("--csv-dir", type=Path, default=root / "data" / "external" / "oceanship_gh")
    ap.add_argument("--out", type=Path, default=root / "out")
    ap.add_argument("--n-seeds", type=int, default=5)
    ap.add_argument(
        "--min-vessels",
        type=int,
        default=3,
        help="drop classes with fewer vessels than this (they cannot populate a 3-way "
        "vessel-disjoint split, so keeping them would give those classes an empty test set)",
    )
    args = ap.parse_args()

    args.out.mkdir(parents=True, exist_ok=True)
    inv, diag = build_inventory(args.csv_dir, args.out)

    ves_per_class: dict[str, set] = collections.defaultdict(set)
    for r in inv:
        ves_per_class[r["label"]].add(r["mmsi"])
    dropped = {
        c: sum(1 for r in inv if r["label"] == c)
        for c in ves_per_class
        if len(ves_per_class[c]) < args.min_vessels
    }
    classes_kept = {c for c in ves_per_class if c not in dropped}

    official_leak = leakage_for(official_assignment(inv), inv)
    split_rows, leak_rows = build_splits(inv, args.n_seeds, classes_kept)
    with gzip.open(args.out / "oceanship_splits.csv.gz", "wt", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=["protocol", "seed", "clip_id", "split"])
        w.writeheader()
        w.writerows(split_rows)

    with open(args.out / "oceanship_leakage_summary.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(leak_rows[0].keys()))
        w.writeheader()
        w.writerows(leak_rows)

    write_report(inv, diag, leak_rows, official_leak, args.out, args, dropped)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /kaggle/working/scripts/w1_02_make_splits.py
#!/usr/bin/env python3
"""W1-02  Generate the evaluation protocols and repeated splits.

This is contribution C2.  The whole point is that UATR papers rarely say which
split they used, and when they do it is usually a random split of CLIPS, which
places clips of the same recording on both sides of the train/test border.  We
generate three protocols under one common definition so that models become
comparable for the first time.

Protocols
---------
P1  clip_random
    Every clip of every recording is pooled and split 70/15/15 at random,
    stratified by class.  This is the protocol behind the familiar
    "DeepShip, 99%" numbers.  It is INCLUDED ON PURPOSE - it is the control
    condition that quantifies the inflation.

P2  rec_random
    Recordings are split 70/15/15 (stratified by class); every clip inherits its
    recording's split.  No recording is ever seen in two splits, so the test set
    measures generalisation to unseen recordings of familiar conditions.

P3  temporal
    Recordings are sorted by acquisition date and cut 70/15/15 chronologically.
    Nothing from the future leaks into training.  This is the deployment-realistic
    protocol.  It is deliberately NOT stratified: the class prior genuinely
    drifts over time, and hiding that would defeat the purpose.

Each protocol is generated for N_SEEDS different random draws wherever there is
randomness, so we can report mean +/- sd and a recording-level bootstrap CI
instead of a single lucky number.  The temporal protocol is deterministic and is
therefore emitted once per seed with identical content - kept for shape
consistency in downstream aggregation.

Output
------
out/splits/splits.csv.gz   one row per (protocol, seed, clip)
                           columns: protocol, seed, rec_key, clip_idx, split
out/splits/split_summary.csv   per (protocol, seed): recordings, clips and
                               class distribution per split - this is paper Table 1.

Reproducibility: the seed is derived from (protocol, seed_index) via a stable
hash, so the files can be regenerated byte-for-byte on any machine.
"""

from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import random
from collections import Counter, defaultdict
from pathlib import Path

PROTOCOLS = ("clip_random", "rec_random", "temporal")
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # test = remainder


def stable_seed(protocol: str, seed_index: int) -> int:
    """Deterministic RNG seed, independent of Python's string hashing."""
    digest = hashlib.sha256(f"{protocol}|{seed_index}".encode()).digest()
    return int.from_bytes(digest[:8], "big")


def stratified_recording_split(
    recs: list[dict], rng: random.Random
) -> dict[str, str]:
    """Split recordings 70/15/15, stratified by class, keeping classes together.

    Returns {rec_key: split}.  Stratifying keeps every class represented in all
    three splits, which matters because DeepShip's classes differ in size and in
    how many recordings they contribute.
    """
    by_class: dict[str, list[str]] = defaultdict(list)
    for r in recs:
        by_class[r["class_dir"]].append(r["rec_key"])

    assign: dict[str, str] = {}
    for _, keys in sorted(by_class.items()):
        keys = sorted(keys)
        rng.shuffle(keys)
        n = len(keys)
        n_train = int(round(n * TRAIN_FRAC))
        n_val = int(round(n * VAL_FRAC))
        # guard against rounding leaving a class with an empty test set
        if n >= 3 and n - n_train - n_val < 1:
            n_val = max(0, n - n_train - 1)
        for i, k in enumerate(keys):
            if i < n_train:
                assign[k] = "train"
            elif i < n_train + n_val:
                assign[k] = "val"
            else:
                assign[k] = "test"
    return assign


def temporal_recording_split(recs: list[dict]) -> dict[str, str]:
    """Chronological 70/15/15 split over recordings, computed PER CLASS.

    Why per class and not globally: DeepShip's classes have almost disjoint
    temporal coverage.  Cargo and Tug were both recorded only in Nov-Dec 2017,
    while Tanker and Passenger span May 2016 - Oct 2018.  A single global sort by
    date therefore pushes every Cargo and Tug recording into the earliest 85%,
    leaving the test set with ZERO Cargo and ZERO Tug clips - a protocol that
    cannot evaluate two of the four classes at all.

    Splitting per class keeps the protocol strictly causal (nothing from the
    future of a class is used to predict that class) while guaranteeing that all
    four classes are represented in every split.  The per-class train/test time
    gaps differ - about one month for Cargo, versus a year-plus for Tanker - which
    is itself useful: it gives the attribution study a range of temporal gaps to
    measure against.
    """
    by_class: dict[str, list[dict]] = defaultdict(list)
    for r in recs:
        by_class[r["class_dir"]].append(r)

    assign: dict[str, str] = {}
    for _, group in sorted(by_class.items()):
        # variant letter orders multiple sessions recorded on the same day,
        # which the metafiles confirm is the chronological order
        ordered = sorted(group, key=lambda r: (r["date"], r["variant"], r["index"]))
        n = len(ordered)
        n_train = int(round(n * TRAIN_FRAC))
        n_val = int(round(n * VAL_FRAC))
        if n - n_train - n_val < 1:
            n_val = max(0, n - n_train - 1)
        for i, r in enumerate(ordered):
            if i < n_train:
                assign[r["rec_key"]] = "train"
            elif i < n_train + n_val:
                assign[r["rec_key"]] = "val"
            else:
                assign[r["rec_key"]] = "test"
    return assign


def main() -> int:
    ap = argparse.ArgumentParser(description="Generate UATR evaluation protocols.")
    ap.add_argument("--recordings", type=Path, required=True, help="deepship_recordings.csv from w1_01")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--n-seeds", type=int, default=5)
    args = ap.parse_args()

    with args.recordings.open(encoding="utf-8") as fh:
        recs = list(csv.DictReader(fh))
    for r in recs:
        r["n_clips"] = int(r["n_clips"])
    print(f"[input] {len(recs)} recordings, {sum(r['n_clips'] for r in recs):,} clips")

    split_dir = args.out / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    split_path = split_dir / "splits.csv.gz"
    summary_path = split_dir / "split_summary.csv"

    summary_rows: list[dict] = []
    n_rows = 0
    class_of = {r["rec_key"]: r["class_dir"] for r in recs}

    def build_clip_random(rng: random.Random) -> list[tuple[str, int, str]]:
        """Pool every clip and split 70/15/15, stratified by class."""
        by_class: dict[str, list[tuple[str, int]]] = defaultdict(list)
        for r in recs:
            for k in range(r["n_clips"]):
                by_class[r["class_dir"]].append((r["rec_key"], k))
        out: list[tuple[str, int, str]] = []
        for _, clips in sorted(by_class.items()):
            rng.shuffle(clips)
            n = len(clips)
            n_train = int(round(n * TRAIN_FRAC))
            n_val = int(round(n * VAL_FRAC))
            if n - n_train - n_val < 1:
                n_val = max(0, n - n_train - 1)
            for i, (rk, k) in enumerate(clips):
                s = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
                out.append((rk, k, s))
        return out

    with gzip.open(split_path, "wt", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["protocol", "seed", "rec_key", "clip_idx", "split"])

        for protocol in PROTOCOLS:
            for seed_index in range(args.n_seeds):
                rng = random.Random(stable_seed(protocol, seed_index))

                if protocol == "clip_random":
                    assignments = build_clip_random(rng)
                    rec_assign = None
                else:
                    rec_assign = (
                        temporal_recording_split(recs)
                        if protocol == "temporal"
                        else stratified_recording_split(recs, rng)
                    )
                    assignments = [
                        (r["rec_key"], k, rec_assign[r["rec_key"]])
                        for r in recs
                        for k in range(r["n_clips"])
                    ]

                for rk, k, s in assignments:
                    writer.writerow([protocol, seed_index, rk, k, s])
                n_rows += len(assignments)

                # ------------------------------------------------------ summary
                counts: Counter = Counter()
                for rk, _, s in assignments:
                    counts[(s, class_of[rk])] += 1

                rec_counts: Counter = Counter()
                if rec_assign is not None:
                    for s in rec_assign.values():
                        rec_counts[s] += 1

                class_totals: Counter = Counter()
                for (_, c), v in counts.items():
                    class_totals[c] += v

                for split in ("train", "val", "test"):
                    row = {
                        "protocol": protocol,
                        "seed": seed_index,
                        "split": split,
                        "recordings": rec_counts.get(split, "") if rec_assign is None else rec_counts.get(split, 0),
                        "clips": sum(v for (s, _), v in counts.items() if s == split),
                    }
                    for c in sorted(class_totals):
                        row[f"clips_{c}"] = counts.get((split, c), 0)
                    for c in sorted(class_totals):
                        tot = class_totals[c]
                        row[f"pct_{c}"] = round(100 * counts.get((split, c), 0) / tot, 1) if tot else 0.0
                    summary_rows.append(row)

    with summary_path.open("w", newline="", encoding="utf-8") as fh:
        fields = list(summary_rows[0].keys())
        w = csv.DictWriter(fh, fieldnames=fields)
        w.writeheader()
        w.writerows(summary_rows)

    # ------------------------------------------------------------------- report
    lines: list[str] = []
    lines.append("=" * 78)
    lines.append("Evaluation protocols")
    lines.append("=" * 78)
    lines.append(f"protocols : {', '.join(PROTOCOLS)}")
    lines.append(f"seeds     : {args.n_seeds} per protocol")
    lines.append(f"split     : {TRAIN_FRAC:.0%} / {VAL_FRAC:.0%} / {1-TRAIN_FRAC-VAL_FRAC:.0%}")
    lines.append(f"rows      : {n_rows:,}")
    lines.append("")
    lines.append("how the three protocols differ")
    lines.append("-" * 78)
    lines.append("  P1 clip_random : clips pooled and shuffled  -> same recording on BOTH sides")
    lines.append("                   (the protocol that produces the 99% headline numbers)")
    lines.append("  P2 rec_random  : recordings shuffled        -> unseen recordings, same conditions")
    lines.append("  P3 temporal    : recordings sorted by date  -> unseen recordings AND unseen future")
    lines.append("")

    by_ps: dict[tuple[str, int], dict[str, dict]] = defaultdict(dict)
    for r in summary_rows:
        by_ps[(r["protocol"], int(r["seed"]))][r["split"]] = r

    for protocol in PROTOCOLS:
        lines.append(f"--- {protocol} ---")
        # average over seeds
        for split in ("train", "val", "test"):
            clips = [by_ps[(protocol, s)][split]["clips"] for s in range(args.n_seeds)]
            recs_ = [by_ps[(protocol, s)][split]["recordings"] for s in range(args.n_seeds)]
            rec_str = f"{sum(recs_)/len(recs_):7.1f}" if isinstance(recs_[0], int) else "      -"
            lines.append(
                f"  {split:<6} clips mean={sum(clips)/len(clips):8,.0f}   recordings mean={rec_str}"
            )
        # class balance drift across splits
        lines.append("  class share per split (%, mean over seeds):")
        classes = sorted({k[len("clips_") :] for k in summary_rows[0] if k.startswith("clips_")})
        hdr = "".join(f"{c:>13}" for c in classes)
        lines.append(f"      {'':<6}{hdr}")
        for split in ("train", "val", "test"):
            vals = []
            for c in classes:
                v = [by_ps[(protocol, s)][split][f"pct_{c}"] for s in range(args.n_seeds)]
                vals.append(f"{sum(v)/len(v):13.1f}")
            lines.append(f"      {split:<6}{''.join(vals)}")
        lines.append("")

    if "temporal" in PROTOCOLS:
        lines.append("NOTE the temporal protocol changes the class prior across splits.")
        lines.append("That drift is part of what the reported 'cross-domain' drop is made of,")
        lines.append("and is normally invisible in a random split. Keep it - do not stratify it away.")
        lines.append("")
        lines.append("temporal protocol: the class prior drifts, and so does the time gap.")
        lines.append("Temporal splitting is done per class because a global sort leaves the")
        lines.append("test set with zero Cargo and zero Tug clips - Cargo and Tug were only")
        lines.append("recorded Nov-Dec 2017 while Tanker and Passenger span 2016-2018.")
        lines.append("")
        tassign = temporal_recording_split(recs)
        by_class_dates: dict[str, dict[str, list[str]]] = defaultdict(lambda: defaultdict(list))
        for r in recs:
            by_class_dates[r["class_dir"]][tassign[r["rec_key"]]].append(r["date"])
        lines.append(f"      {'class':<11}{'train span':<20}{'val span':<20}{'test span':<20}{'gap':>6}")
        for cls in sorted(by_class_dates):
            d = by_class_dates[cls]
            def span(s: str) -> str:
                v = sorted(d.get(s, []))
                return f"{v[0]}..{v[-1]}" if v else "-"
            tr, te = sorted(d.get("train", [])), sorted(d.get("test", []))
            gap = ""
            if tr and te:
                from datetime import date as _d
                a = _d(int(tr[-1][:4]), int(tr[-1][4:6]), int(tr[-1][6:]))
                b = _d(int(te[0][:4]), int(te[0][4:6]), int(te[0][6:]))
                gap = f"{(b - a).days}d"
            lines.append(f"      {cls:<11}{span('train'):<20}{span('val'):<20}{span('test'):<20}{gap:>6}")
        lines.append("")
        lines.append("      The 'gap' column is train-end to test-start: the temporal distance the")
        lines.append("      model has to extrapolate across. It differs per class, which gives the")
        lines.append("      attribution study several distinct temporal gaps to compare.")
        lines.append("")

    lines.append(f"written: {split_path}")
    lines.append(f"written: {summary_path}")

    text = "\n".join(lines)
    print()
    print(text)
    (split_dir / "w1_02_protocol_report.txt").write_text(text, encoding="utf-8")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# --------------------------------------------------------------------------- #
# 4. resolve the datasets and the files the builder needs
#
# Two rules, both learned from a failed run:
#   * never enumerate the audio folder by pattern - one directory there holds
#     53,770 entries and a glob that reaches into it costs ~4 minutes;
#   * never die on a missing file - report it, degrade, keep building.
# --------------------------------------------------------------------------- #
import gzip
import os
import re
import time

T0 = time.time()


def tick(msg):
    print(f"  [{time.time() - T0:6.1f}s] {msg}", flush=True)


_LIST = {}


def listing(d):
    """One iterdir per folder, cached. Never recursive."""
    key = str(d)
    if key not in _LIST:
        try:
            _LIST[key] = sorted(d.iterdir(), key=lambda p: p.name)
        except Exception as exc:
            _LIST[key] = []
            print(f"  [warn] cannot list {d}: {exc}")
    return _LIST[key]


def descend(d, hops=3):
    """A dataset uploaded as a .zip is extracted one folder deeper.

    Never descends into the audio folder itself: a root that holds *only*
    `3090latest_wav/` is a perfectly normal mount, and descending into it would
    then enumerate 53,770 entries - the exact four-minute mistake this version
    exists to avoid.
    """
    for _ in range(hops):
        kids = listing(d)
        dirs = [k for k in kids if k.is_dir()]
        if (len(kids) == 1 and len(dirs) == 1
                and not dirs[0].name.startswith("3090latest_wav")):
            print(f"  ... {d} holds only a folder, descending into {dirs[0].name}/")
            d = dirs[0]
        else:
            break
    return d


def resolve(slug, *alts):
    if MANUAL.get(slug):
        return Path(MANUAL[slug])
    names = {slug, *alts}
    for d in MOUNTED:
        if d.name in names:
            return d
    return None


DEEP = resolve("deepship-raw")
OCEAN = resolve("oceanship")
WS = resolve("workspace", "uwatr-workspace", "uwatr_workspace", "workspace-uwatr")

missing_ds = [n for n, d in (("deepship-raw", DEEP), ("oceanship", OCEAN),
                             ("workspace", WS)) if d is None]
if missing_ds:
    print()
    print("  MISSING DATASET(S):", missing_ds)
    print("  -> Add Input, tick them, re-run this cell. If one is attached under")
    print("     another name, copy its path from the listing above into MANUAL:")
    print('       MANUAL["workspace"] = "/kaggle/input/datasets/<owner>/<slug>"')
    raise SystemExit("dataset not attached: " + ", ".join(missing_ds))

DEEP, OCEAN, WS = descend(DEEP), descend(OCEAN), descend(WS)
print(f"deepship-raw    : {DEEP}")
print(f"oceanship       : {OCEAN}")
print(f"workspace       : {WS}")
tick("datasets resolved")

# ---- Oceanship audio: archive, extracted folder, or raw split chunks -------- #
AUDIO_CHUNK = re.compile(r".*3090latest_wav\d\d$")
CLASSES = ("Cargo", "Passenger", "Tanker", "Tug")


def classify(d):
    """What audio delivery is in this folder? Never opens the audio folder."""
    kids = listing(d)
    zips = [k for k in kids if k.is_file() and k.suffix.lower() == ".zip"]
    if zips:
        zips.sort(key=lambda p: -p.stat().st_size)
        return zips[0], f"archive {zips[0].name} ({zips[0].stat().st_size / 1e9:.2f} GB)"
    wavdirs = [k for k in kids if k.is_dir() and k.name.startswith("3090latest_wav")]
    if wavdirs:
        return wavdirs[0], f"extracted wav folder {wavdirs[0].name}/"
    if sum(1 for k in kids if k.is_file() and AUDIO_CHUNK.match(k.name)) >= 2:
        return d, f"split chunk set in {d.name}/"
    return None


hit = classify(OCEAN)
if hit is None:
    for sub in listing(OCEAN):
        if not sub.is_dir() or sub.name.startswith("3090latest_wav"):
            continue           # <- never enumerate the 53,770-entry audio folder
        hit = classify(sub)
        if hit:
            break
OCEAN_AUDIO, AUDIO_WHAT = hit if hit else (OCEAN, "raw root (the builder decides)")
print(f"oceanship audio : {AUDIO_WHAT}")
print(f"                  {OCEAN_AUDIO}")

# ---- DeepShip: the folder that actually holds Cargo/ etc. ------------------ #
ds_root = DEEP if any((DEEP / c).is_dir() for c in CLASSES) else None
if ds_root is None:
    for sub in listing(DEEP):
        if sub.is_dir() and any((sub / c).is_dir() for c in CLASSES):
            ds_root = sub
            break
ds_root = ds_root or DEEP
print(f"deepship root   : {ds_root}")
print(f"  class folders : {[c.name for c in listing(ds_root) if c.is_dir()]}")
tick("audio roots resolved")

# ---- the workspace files --------------------------------------------------- #
def walk_ws(root, budget=20000, max_depth=5):
    """One bounded walk. Returns (files, dirs, listing, examined).

    Both files and folders are indexed, because Kaggle expands archives on
    ingest: `ShipsEar.zip` can arrive as the folder `shipsear/ShipsEar/`.
    """
    files, dirs, shown, seen, stack = {}, {}, [], 0, [(root, 0)]
    while stack:
        d, depth = stack.pop()
        for c in listing(d):
            seen += 1
            if seen > budget:
                print(f"  [warn] stopped walking {root.name} after {budget:,} entries")
                return files, dirs, shown, seen
            try:
                rel = str(c.relative_to(root)).replace("\\", "/")
            except ValueError:
                rel = c.name
            if c.is_file():
                files.setdefault(c.name.lower(), c)
                shown.append((rel, c.stat().st_size, "gz" if rel.endswith(".gz") else ""))
            else:
                dirs.setdefault(c.name.lower(), c)
                shown.append((rel + "/", 0, "dir"))
                if depth + 1 <= max_depth:
                    stack.append((c, depth + 1))
    return files, dirs, shown, seen


WSF, WSD, WS_LIST, WS_N = walk_ws(WS)
print(f"workspace       : walked {WS_N:,} entries -> {len(WSF):,} files, "
      f"{len(WSD):,} folders")
tick("workspace walked")

DATA = (".csv", ".gz", ".zip", ".json", ".txt")


def pick(*exact, contains=()):
    """Exact name first, then the same name without .gz, then a contains-match.

    Kaggle expands `x.csv.gz` into `x.csv` on ingest, so every lookup has to
    accept both spellings or the file looks missing when it is not.
    """
    for name in exact:
        for cand in (name, name[:-3] if name.endswith(".gz") else None):
            if cand and cand.lower() in WSF:
                return WSF[cand.lower()]
    for key in sorted(WSF):
        if all(t in key for t in contains) and key.endswith(DATA):
            return WSF[key]
    return None


def pick_dir(contains):
    for key in sorted(WSD):
        if all(t in key for t in contains):
            return WSD[key]
    return None


INV = pick("oceanship_inventory.csv.gz", contains=("inventory",))
OSPLIT = pick("oceanship_splits.csv.gz", contains=("oceanship", "splits"))
OLEAK = pick("oceanship_leakage_summary.csv", contains=("oceanship", "leakage"))
RECS = pick("deepship_recordings.csv", contains=("recordings",))
DSPLIT = pick("splits.csv.gz")
DSSUM = pick("split_summary.csv")
FGTRAIN = pick("oceanship_fg_train.csv", contains=("fg_train",))
FGTEST = pick("oceanship_fg_test.csv", contains=("fg_test",))

# ShipsEar: the archive, or the tree Kaggle extracted it into
SEA = pick("shipsear.zip", contains=("shipsear",))
SEA_FORM = "archive" if SEA is not None else ""
if SEA is None:
    SEA = pick_dir(("shipsear",))
    SEA_FORM = "expanded folder" if SEA is not None else ""

# ---- regenerate every derived table, and check the mounted ones against it -- #
# The paper's whole claim is that this protocol is reproducible, and that is only
# true if every split in the artifact comes out of the generator. So the
# generators are run here and the mounted files are compared with them BY
# CONTENT (gzip headers make byte sizes useless as a fingerprint). Anything that
# disagrees is replaced, loudly: it means the mounted file came from an older
# draw, and this project's own history has one - w1_04 used to seed from the
# builtin hash(), so the same CSVs produced a different "frozen" split per
# process. The artifact must not inherit that.
WSEFF = Path("/kaggle/working/ws")
WSEFF.mkdir(parents=True, exist_ok=True)


def stage(sub, src):
    """Put a chosen file where the builder looks, under the name it arrived as.

    The builder resolves both spellings and re-compresses on the way into the
    artifact, so nothing here needs to normalise anything by hand.
    """
    if src is None:
        return False
    src = Path(src)
    dst = WSEFF / sub / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return True
    try:
        os.link(src, dst)          # same filesystem: free
    except OSError:
        shutil.copy2(src, dst)
    return True


def read_text(path) -> str:
    """Text of a .csv or .csv.gz, decided by magic bytes rather than the name."""
    path = Path(path)
    with open(path, "rb") as fh:
        gz = fh.read(2) == b"\x1f\x8b"
    opener = gzip.open if gz else open
    with opener(path, "rt", newline="", encoding="utf-8") as fh:
        return fh.read()


REBUILT = Path("/kaggle/working/ws_rebuilt")
REBUILT.mkdir(parents=True, exist_ok=True)

GEN_OCEAN = None
if FGTRAIN is not None and FGTEST is not None and FGTRAIN.parent == FGTEST.parent:
    print()
    print("  regenerating the Oceanship inventory and splits from the official CSVs")
    cmd = [sys.executable, str(SCRIPTS / "w1_04_oceanship_inventory.py"),
           "--csv-dir", str(FGTRAIN.parent), "--out", str(REBUILT / "oceanship")]
    print("  $ " + " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    GEN_OCEAN = REBUILT / "oceanship"
    tick("oceanship tables regenerated")
else:
    print()
    print("  [note] the official oceanship_fg_*.csv are not both present, so the")
    print("         Oceanship split cannot be regenerated; whatever is mounted is used")

GEN_DEEP = None
if RECS is not None:
    print("  regenerating the DeepShip splits from the recording table")
    cmd = [sys.executable, str(SCRIPTS / "w1_02_make_splits.py"),
           "--recordings", str(RECS), "--out", str(REBUILT / "deepship")]
    print("  $ " + " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    GEN_DEEP = REBUILT / "deepship" / "splits"
    tick("deepship split regenerated")
else:
    print("  [note] no deepship_recordings.csv, so the DeepShip split cannot be")
    print("         regenerated; whatever is mounted is used")


def reconcile(label, generated, mounted, sub):
    """Use the generator's output; keep the mounted copy only when it agrees."""
    gen = Path(generated) if generated is not None else None
    if gen is None or not gen.exists():
        if mounted is not None:
            stage(sub, mounted)
        return mounted
    if mounted is not None and read_text(mounted) == read_text(gen):
        print(f"  [{label}] verified reproducible - the mounted file is exactly")
        print("              what the generator produces")
        stage(sub, mounted)
        return mounted
    why = "not present" if mounted is None else "DIFFERS from the generator"
    print(f"  [{label}] {why} -> using the regenerated file")
    if mounted is not None:
        a = read_text(mounted).splitlines()
        b = read_text(gen).splitlines()
        diff = sum(1 for x, y in zip(a, b) if x != y)
        print(f"              mounted {len(a):,} rows, regenerated {len(b):,} rows, "
              f"{diff:,} of the common rows disagree")
    stage(sub, gen)
    return WSEFF / sub / gen.name


print()
print("=" * 88)
print("reproducibility of the frozen protocol")
print("=" * 88)
INV = reconcile("inventory", GEN_OCEAN / "oceanship_inventory.csv.gz" if GEN_OCEAN else None,
                INV, "metadata")
OSPLIT = reconcile("oceanship splits",
                   GEN_OCEAN / "oceanship_splits.csv.gz" if GEN_OCEAN else None,
                   OSPLIT, "splits")
OLEAK = reconcile("oceanship leakage",
                  GEN_OCEAN / "oceanship_leakage_summary.csv" if GEN_OCEAN else None,
                  OLEAK, "splits")
DSPLIT = reconcile("deepship splits",
                   GEN_DEEP / "splits.csv.gz" if GEN_DEEP else None, DSPLIT, "splits")
DSSUM = reconcile("deepship summary",
                  GEN_DEEP / "split_summary.csv" if GEN_DEEP else None, DSSUM, "splits")
tick("workspace files resolved")

# ---- report, and degrade instead of aborting ------------------------------- #
print()
print("=" * 88)
print("inputs")
print("=" * 88)
for label, p, form in (("oceanship inventory", INV, "table"),
                       ("oceanship splits", OSPLIT, "table"),
                       ("deepship recordings", RECS, "table"),
                       ("deepship splits", DSPLIT, "table"),
                       ("ShipsEar", SEA, SEA_FORM)):
    if p is None:
        print(f"  {label:<22}: MISSING")
    else:
        note = ""
        if form == "expanded folder":
            note = "  <- Kaggle expanded ShipsEar.zip on ingest; read from the tree"
        elif p.suffix != ".gz" and p.name.endswith(".csv"):
            note = "  <- arrived expanded (no .gz)"
        print(f"  {label:<22}: {p}{note}")

print()
print(f"  everything under {WS}:")
for rel, size, kind in sorted(WS_LIST)[:120]:
    print(f"    {kind:<4} {size:>13,}  {rel}" if kind != "dir" else f"    dir  {'':>13}  {rel}")

DEGRADED = [n for n, p in (("oceanship inventory", INV), ("oceanship splits", OSPLIT),
                           ("deepship recordings", RECS), ("deepship splits", DSPLIT),
                           ("ShipsEar", SEA)) if p is None]
if DEGRADED:
    print()
    print("!" * 88)
    print(f"  DEGRADED - missing: {DEGRADED}")
    print("  the run continues and the corpora that need them are skipped")
    print("!" * 88)
    print("  to restore them: re-create the Kaggle dataset 'workspace' from")
    print("  upload/uwatr-workspace (56 files, 314.6 MiB).")
    print()

if INV is None:
    print()
    print("  STOP: with no Oceanship inventory there is no label, no MMSI and no")
    print("  vessel-disjoint split for the primary corpus, so there is nothing")
    print("  worth building. Fix the workspace dataset and re-run this cell.")
    raise SystemExit("workspace dataset unusable")

print(f"  free space in /kaggle/working: {shutil.disk_usage(OUT).free / 1e9:.1f} GB")
tick("cell done")


## Why this is one script and not three

Oceanship, DeepShip and ShipsEar differ in container, sample rate and labelling,
but they have to land in the *same* feature space or the cross-corpus results
compare pipelines instead of data. So there is one set of STFT / filterbank / dB
constants and every job uses it:

| | |
|---|---|
| sample rate | 16 kHz mono (single polyphase resampler for all corpora) |
| rows | 5 s, 80,000 samples |
| log-mel | (N, 128, 313) fp16, 20-8000 Hz, Slaney area normalisation |
| linear | (N, 128, 313) fp16, 513 bins pooled into 128 bands |
| scaling | dB relative to each clip's peak, floored at -80 dB |

Two representations on purpose: if the strict-protocol collapse shows up in mel
only, that is a feature-engineering artefact; if it shows up in both, it is a
property of the data. The paper needs the second statement, so the ablation is
built in from the start.

Short clips are **padded, not dropped**, and every row's measured source
duration goes into `index.csv` as `src_dur_s`. That column exists because
duration is class-correlated in Oceanship (Passenger median 3.67 s against Tug
5.00 s), so padding without recording it would hand the model a silent shortcut.
With the column, a with/without-short-clips ablation is one join away.

## Next


In [ ]:
# --------------------------------------------------------------------------- #
# 5. probe: headers only, writes nothing
#
# A plumbing check: are the paths right, does every corpus decode, does the class
# coverage look sane? A full pass here costs 10-13 minutes and cell 6 prints the
# same tables anyway, so this runs on a sample. Set PROBE_LIMIT = 0 for the full
# measurement when the numbers themselves are what you want.
#
# Runs through subprocess rather than shell magic: no line-continuation quirks,
# and a non-zero exit stops the notebook instead of quietly producing half a
# dataset.
# --------------------------------------------------------------------------- #
PROBE_LIMIT = 3000        # 0 = measure every clip (slow; cell 6 does it anyway)

BUILD = str(SCRIPTS / "build_features.py")
COMMON = ["--out", str(OUT), "--workspace", str(WSEFF),
          "--oceanship-audio", str(OCEAN_AUDIO), "--deepship-root", str(ds_root)]
for flag, path in (("--oceanship-inventory", INV), ("--deepship-recordings", RECS),
                   ("--shipsear-zip", SEA)):
    if path is not None:
        COMMON += [flag, str(path)]

cmd = [sys.executable, BUILD, "--probe", "--job", "all"]
if PROBE_LIMIT:
    cmd += ["--limit", str(PROBE_LIMIT)]
cmd += COMMON
print("$ " + " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
# --------------------------------------------------------------------------- #
# 6. build the cache
#
# --device cpu on purpose: the cache is a public artifact and must not depend on
# which accelerator was allocated. Switch to cuda only if you accept that the
# frozen bytes may then differ from a CPU build of the same audio.
#
# The script now measures every corpus first, checks free space ONCE, and only
# then writes. If the check fails it says so with nothing written, so a re-run
# starts clean instead of leaving half a cache behind.
#
# Space: mel+spec is about 14.4 GB of the ~20 GB working directory.
# --------------------------------------------------------------------------- #
PREVIOUS = False          # set True to delete a previous attempt's arrays first

stale = sum(p.stat().st_size for p in OUT.rglob("*.npy")) if OUT.exists() else 0
if stale:
    print(f"  NOTE: {stale / 1e9:.2f} GB of arrays from a previous attempt are in {OUT}")
    print("  they will be overwritten, but they still occupy space. If the space")
    print("  check aborts, remove them first (this is Kaggle's scratch dir, safe to")
    print(f"  delete: the input datasets are read-only and unaffected):")
    print(f"      !rm -rf {OUT}")
    print()
if PREVIOUS and OUT.exists():
    shutil.rmtree(OUT)
    print(f"  PREVIOUS=True: removed {OUT}")

cmd = [sys.executable, BUILD, "--job", "all", "--device", "cpu",
       "--reps", "mel+spec"] + COMMON
print("$ " + " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
# --------------------------------------------------------------------------- #
# 7. verify the artifact, and prove the split join actually works
# --------------------------------------------------------------------------- #
import csv
import gzip
import json

OUT = Path("/kaggle/working/uwatr-features")
man = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))
print("=" * 88)
print("manifest")
print("=" * 88)
print(json.dumps(man["jobs"], indent=2))
print("splits copied:", man.get("splits_copied"))
if DEGRADED:
    print("DEGRADED this run (missing inputs):", DEGRADED)

print()
print("=" * 88)
print("arrays")
print("=" * 88)
for job in ("oceanship", "deepship", "shipsear"):
    for rep in ("logmel.npy", "spec.npy"):
        p = OUT / job / rep
        if not p.exists():
            continue
        a = np.load(p, mmap_mode="r")
        head = np.asarray(a[: min(200, a.shape[0]), :, :], dtype=np.float32)
        print(f"  {job}/{rep:<11} shape={tuple(a.shape)} dtype={a.dtype} "
              f"range=[{head.min():.2f}, {head.max():.2f}]  {p.stat().st_size/1e9:.2f} GB")
        del a, head


def header(path):
    op = gzip.open if path.suffix == ".gz" else open
    with op(path, "rt", newline="", encoding="utf-8") as fh:
        return next(csv.reader(fh))


def rows(path):
    op = gzip.open if path.suffix == ".gz" else open
    with op(path, "rt", newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


print()
print("=" * 88)
print("split join check - every protocol that exists, every seed")
print("=" * 88)
# The two corpora do NOT share protocol names: Oceanship has vessel_disjoint,
# DeepShip has temporal. Hardcoding one name silently reports zero rows for the
# other corpus, so the protocols are read out of each file instead.
for job in ("oceanship", "deepship"):
    idx_path = OUT / job / "index.csv"
    sp_path = OUT / "splits" / f"{job}_splits.csv.gz"
    if not (idx_path.exists() and sp_path.exists()):
        print(f"  {job}: skipped (no index or no splits)")
        continue
    ih, sh = header(idx_path), header(sp_path)
    keys = [c for c in ih if c in sh and c not in ("protocol", "seed", "split")]
    if not keys:
        print(f"  {job}: no shared join key between index and splits")
        continue
    have = {tuple(r[k] for k in keys) for r in rows(idx_path)}
    allsp = rows(sp_path)
    protocols = sorted({r["protocol"] for r in allsp})
    print(f"  {job}: key {keys}   cache rows {len(have):,}   "
          f"split rows {len(allsp):,}   protocols {protocols}")
    for proto in protocols:
        for seed in sorted({r["seed"] for r in allsp if r["protocol"] == proto},
                           key=int):
            counts, matched = {}, 0
            for r in allsp:
                if r["protocol"] != proto or r["seed"] != seed:
                    continue
                if tuple(r[k] for k in keys) in have:
                    matched += 1
                    counts[r["split"]] = counts.get(r["split"], 0) + 1
            print(f"    {proto:<16} seed {seed}  joined {matched:>6,} / {len(have):,}"
                  f"  -> {counts}")

print()
print("=" * 88)
print("test size is a random variable under unit-level protocols - report the spread")
print("=" * 88)
for job in ("oceanship", "deepship"):
    sp_path = OUT / "splits" / f"{job}_splits.csv.gz"
    if not sp_path.exists():
        continue
    allsp = rows(sp_path)
    for proto in sorted({r["protocol"] for r in allsp}):
        if proto == "clip_random":
            continue          # clip-level: test size is fixed by construction
        per_seed = {}
        for r in allsp:
            if r["protocol"] == proto and r["split"] == "test":
                per_seed[r["seed"]] = per_seed.get(r["seed"], 0) + 1
        if not per_seed:
            continue
        sizes = sorted(per_seed.values())
        spread = 100 * (max(sizes) - min(sizes)) / (sum(sizes) / len(sizes))
        print(f"  {job}/{proto}: {per_seed}")
        print(f"    min {min(sizes):,}  max {max(sizes):,}  "
              f"mean {sum(sizes)/len(sizes):,.0f}  spread {spread:.0f}% of the mean")

print()
tot = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())
print(f"total output: {tot/1e9:.2f} GB under {OUT}")
print()
print("next: Output tab -> Create Dataset -> pick the whole uwatr-features folder")


## How the inputs are found, and the one thing that kept breaking

**Kaggle expands archives when it ingests a dataset.** That single fact explains
every input failure so far:

| uploaded as | arrives as |
|---|---|
| `uwatr-workspace.zip` | `workspace/uwatr-workspace/...` — one folder deeper |
| `3090latest_wav.zip` | `oceanship/3090latest_wav/…wav` — 53,770 loose files |
| `ShipsEar.zip` | `shipsear/ShipsEar/shipsear_5s_16k/<class>/<group>/<file>.wav` |
| `oceanship_inventory.csv.gz` | `metadata/oceanship_inventory.csv` |
| `oceanship_splits.csv.gz` | `splits/oceanship_splits.csv` |

So the notebook and the builder both accept **either spelling or form**:

1. **A dataset root holding exactly one folder is descended into.**
2. **The audio folders are never enumerated by pattern.** `3090latest_wav/` holds
   53,770 entries; a glob that reaches into it cost about four minutes on
   Kaggle's mount. Detection classifies the *parent* listing and never lists the
   audio folder itself.
3. **`x.csv.gz` and `x.csv` are both accepted**, by magic-byte sniffing rather
   than by name.
4. **ShipsEar is read from the archive or from the extracted tree.** Matching on
   the last four path components means both produce the *same* `clip_id`.

## The frozen protocol is regenerated and checked, not trusted

The artifact's `splits/` must be reproducible from the published generators, so
cell 4 **runs both generators** and compares their output with the mounted files
**by content** — gzip headers make byte sizes useless as a fingerprint:

| generator | input | produces |
|---|---|---|
| `w1_04_oceanship_inventory.py` | `metadata/oceanship_fg_{train,test}.csv` | inventory + Oceanship splits + leakage |
| `w1_02_make_splits.py` | `deepship/deepship_recordings.csv` | DeepShip splits + summary |

A file that agrees is kept and reported as *verified reproducible*. A file that
disagrees is **replaced**, loudly — that is how the one non-reproducible draw in
this project's history would have been caught: `w1_04` used to seed from the
builtin `hash()`, which `PYTHONHASHSEED` salts per process, so the same CSVs
produced a different "frozen" split every run.

If the official CSVs are absent the split cannot be regenerated, the cell says
so, and whatever is mounted is used as-is.

## What is fatal

Only having no inventory at all. Missing ShipsEar drops that corpus; a missing
inventory/splits is regenerated. If a dataset is attached under another name,
copy its path from the listing in cell 1 into `MANUAL`.


## What the next notebook needs

```python
import numpy as np, pandas as pd

FEAT = "/kaggle/input/uwatr-features"     # <- the dataset you just created

logmel = np.load(f"{FEAT}/oceanship/logmel.npy", mmap_mode="r")   # (N,128,313) fp16
index  = pd.read_csv(f"{FEAT}/oceanship/index.csv")
splits = pd.read_csv(f"{FEAT}/splits/oceanship_splits.csv.gz")

# vessel_disjoint is the only protocol that clears both leakage tests
s = splits.query("protocol == 'vessel_disjoint' and seed == 0")
index["split"] = index.clip_id.map(s.set_index("clip_id")["split"])

# logmel[index.split == 'train'] is now a memmap slice - no RAM copy
```

For DeepShip join on `(rec_key, clip_idx)` against `splits/deepship_splits.csv.gz`.

The `index.csv` also carries `mmsi`, `label`, `official_split`, `src_dur_s` and
the AIS fields, so the metadata-conditioned domain split is another join, not a
re-derivation.

## A note on the vessel-disjoint test set

Because one vessel contributes anything from a handful to hundreds of clips, the
vessel-disjoint test set is a random variable: across the five seeds it ranges
from 6,442 to 8,531 clips (mean 7,587, plus or minus 24%). That is a property of
the protocol, not a bug. Report per-seed results and the spread, never a single
seed's accuracy as *the* number — an unlucky seed can move a percentage point on
test size alone.
